# Post-processing of variant calls

This notebook turns the per-timepoint output of the *in silico* noise-correction model into the final
somatic and germline variant call tables used throughout the rest of this repository and published as
**Supplementary Tables 6 and 7**. It is the step between the TETRIS-seq pipeline and every figure
notebook here.

The filtering follows the Supplementary Methods:

1. **Rescue** variants called as errors at one timepoint (p < 0.1) where the same variant was called as
   real (p < 1e-10) at another timepoint in the same individual. Note that where VAF > 10% the error
   model returns no p-value; these are read in as 0, so such a timepoint always satisfies the
   p < 1e-10 condition and the variant is called real on that basis.
2. **Remove overly common variants**, separately for non-germline and germline calls.
3. **Remove likely error trajectories** (SNVs and indels) using:
   - sample-level mean DCS depth across the panel >= 100;
   - the variant must cause an amino acid change (`AA_change` is not `.`);
   - per-position total depth >= 500, or within 2 s.d. of that sample's mean panel depth;
   - strand bias phred < 60, mean position in read > 8, non-synonymous changes only, and >= 2 variant
     reads (*FLT3*-ITD exempt);
   - for a variant missing from timepoints **between** two detections, the growth rate implied by the
     flanking VAFs, used to ask how likely it was to have been missed at each (eqs. 1-3 of the
     Supplementary Methods);
   - for a variant detected at a **single, middle** timepoint, where no growth rate can be estimated:
     the larger of the two probabilities of having missed it at the preceding and following timepoints,
     given the observed VAF and the read depth there. Under any monotonic trajectory the clone must have
     been at least as large as the observed VAF at one of the two, so this is a bound that needs no
     assumption about growth rate. First and last timepoints are exempt -- a declining clone has no
     earlier constraint, an emerging one no later constraint. Threshold:
     `single_timepoint_chance_threshold`.

   mCAs are not filtered here: they are read as called, minus rows marked `not processed` or
   `contaminated?`. ExAC frequency is applied at the germline step (4), not here.
4. **Remove likely germline variants** (EXAC frequency, VAF near 0.5 or 1 across all timepoints).
5. **Exclude contaminated timepoints** identified from the trajectories.
6. **Plot** each individual's variant trajectories.
7. **Write** the per-timepoint post-processed call files and the combined cohort tables.

## Where the inputs come from

| input | produced by |
|---|---|
| `Beta_binomial_variant_calls/DCS/*_beta_binomial_SNV_all_variant_calls_Oct_2023.txt` | the position-specific beta-binomial error model. The model itself is in this repository, in `Supplementary_Fig_12.ipynb` (`BB_likelihood`, `estimate_delta_method_of_moments`, `beta_binomial_MLE` and the calling wrappers), where it is fitted to the SLX_20124 development lane. The same model is run across every lane at production scale by `Duplex_Error_Model_initial_variant_calling_v5.py` in the [TETRIS-seq](https://github.com/the-blundell-lab/TETRIS-seq) repository, which is what produced these per-timepoint files |
| `Beta_binomial_variant_calls/VarDictJava_annotated/*_VarDictJava_annotated.txt` | VarDictJava indel calling in the TETRIS-seq pipeline, annotated with ANNOVAR. The per-timepoint indel call files (`VarDictJava_indels/*_indel_variant_calls.txt`) are written from these by the first section of this notebook |
| `VCF_files/DCS/*_all_positions.vcf` | the TETRIS-seq pipeline; used here only for read depth at a position |
| `mCA_calls/*_mCA.txt` | the mCA callers (see the Supplementary Fig. 26–38 notebooks) |
| `UKCTOCS_samples_processed_information.csv` | the UKCTOCS sample sheet — case/control status, sample and diagnosis ages, time to diagnosis and matched pairs. It defines the 93-participant cohort this notebook loops over; on EGA |

## Data availability

The per-timepoint inputs are individual-level participant data and are **not distributed with this
code**. They are deposited under controlled access in the European Genome-phenome Archive (EGA) and
released to approved researchers via a Data Access Committee administered by the corresponding
authors: the raw reads, the all-positions VCFs, the annotated DCS variant tables, the post-error-model
beta-binomial SNV call files read below, and the annotated VarDictJava output that the indel calls are
built from. `UKCTOCS_samples_processed_information.csv` is likewise individual-level
participant data, available from the EGA under the same controlled access; it supplies the cohort
definition, sample and diagnosis ages and matched case–control pairs.

## Outputs

Ages and event timings are given to full precision in the files used for analysis, and reduced to
whole completed units (rounded down, so an age of 73.73 years is reported as 73) in the files
intended for publication.

| output | contents | ages | used by |
|---|---|---|---|
| `Data_files/UKCTOCS_non-germline_variants_calls_SNVs_indels_mCAs.csv` | somatic SNVs, indels, *FLT3*-ITDs and mCAs | full precision | `Figure_1`, `Figure_3d-g`, `Code_for_inferring_acquisition_age_and_fitness_v16.py`; also the EGA deposit |
| `Data_files/UKCTOCS_non-germline_variants_calls_SNVs_indels_mCAs_rounded_ages.csv` | the same calls | rounded | `Extended_Data_Figure_2` |
| `Data_files/Somatic_SNV_indel_FLT3_calls.csv` | the same calls without the mCAs | rounded | **Supplementary Table 6** |
| `Data_files/UKCTOCS_germline_variants_calls_SNV_indel_panel.csv` | germline SNVs and indels | full precision | `Code_for_inferring_acquisition_age_and_fitness_v16.py` and `Figure_3d-g`, which annotate potentially relevant germline variants on the trajectory plots; also the EGA deposit |
| `*_non-germline_variant_calls_2026_post_processed.txt` and the germline/indel equivalents | one file per timepoint, split germline / non-germline | as the table above | Supplementary Fig. 15 panel b |

**Supplementary Table 7** (mCA calls) is not produced here: it is the output of the mCA caller, and is
in this repository as `Data_files/Somatic_mCA_calls.csv`. The mCA calls are read in by this notebook
only so that they can be included in the combined call table and in the trajectory plots.


In [ ]:
# imported packages
import csv
import re
import math
import numpy as np
import matplotlib.pyplot as plt
import scipy.special
import os
import pandas as pd
from matplotlib.ticker import MultipleLocator

In [ ]:
# Input locations (see the data availability note above).
SNV_DIR   = 'Data_files/Beta_binomial_variant_calls/DCS/'
ANNOTATED_INDEL_DIR = 'Data_files/Beta_binomial_variant_calls/VarDictJava_annotated/'
INDEL_DIR = 'Data_files/Beta_binomial_variant_calls/VarDictJava_indels/'
VCF_DIR   = 'Data_files/VCF_files/DCS/'
MCA_DIR   = 'Data_files/mCA_calls/'

for label, d in [('beta-binomial SNV calls', SNV_DIR), ('VarDictJava annotated output', ANNOTATED_INDEL_DIR),
                 ('all-positions VCFs', VCF_DIR), ('mCA calls', MCA_DIR)]:
    if not os.path.isdir(d):
        print(f'missing input directory: {d}')
    else:
        n = len([x for x in os.listdir(d) if not x.startswith('.')])
        print(f'{label:<24} {n:>5} files')

In [ ]:
# Lists of colors for plots
c0 = (0.76, 0.76, 0.76)
c1 = (1.00, 0.18, 0.33);
c2 = (1.00, 0.23, 0.19);
c3 = (1.00, 0.58, 0.00);
c4 = (1.00, 0.80, 0.00);
c5 = (0.30, 0.85, 0.39);
c6 = (0.35, 0.78, 0.98);
c7 = (0.20, 0.67, 0.86);
c8 = (0.00, 0.48, 1.00);
c9 = (0.35, 0.34, 0.84);
c10 = (0.00, 0.31, 0.57);
c11 = (0.12, 0.29, 0.69);
c12 = (0.17, 0.17, 0.42);
c13 = (1.00, 1.00, 1.00);
c14 = (0.77, 0.04, 0.00);

In [ ]:
#define the colors from colorbrewer2
orange1 = '#feedde'
orange2 = '#fdbe85'
orange3 = '#fd8d3c'
orange4 = '#e6550d'
orange5 = '#a63603'
blue1 = '#eff3ff'
blue2 = '#bdd7e7'
blue3 = '#6baed6'
blue4 = '#3182bd'
blue5 = '#08519c'
green1 = '#edf8e9'
green2 = '#bae4b3'
green3 = '#74c476'
green4 = '#31a354'
green5 = '#006d2c'
grey1 = '#f7f7f7'
grey2 = '#cccccc'
grey3 = '#969696'
grey4 = '#636363'
grey5 = '#252525'
purple1 = '#f2f0f7'
purple2 = '#cbc9e2'
purple3 = '#9e9ac8'
purple4 = '#756bb1'
purple5 = '#54278f'
red1 = '#fee5d9'
red2 = '#fcae91'
red3 = '#fb6a4a'
red4 = '#de2d26'
red5 = '#a50f15'
yellow = '#ffffd4'

In [ ]:
cb_blue1 = '#a6cee3'
cb_blue2 = '#1f78b4'
cb_green1 = '#b2df8a'
cb_green2 = '#33a02c'
cb_pink = '#fb9a99'
cb_red = '#e31a1c'
cb_orange1 = '#fdbf6f'
cb_orange2 = '#ff7f00'
cb_purple1 = '#cab2d6'
cb_purple2 = '#6a3d9a'

In [ ]:
# plt.rcParams['axes.prop_cycle'] = plt.cycler(color=plt.cm.Set3.colors)
plt.rcParams['axes.prop_cycle'] = plt.cycler(color=[blue4, orange3, purple4, red4, green4, cb_blue1, cb_orange1, cb_purple2, cb_pink, cb_green2]) 

In [ ]:
mutation_class_text_colors = {'NPM1': c1,
                        'DNA methylation': '#3485A5',
                        'Chromatin modifiers': '#E06F22',
                        'Transcription factors':  '#4DA04A',
                        'Transcriptional corepressors':  '#2A7726',
                        'Tumour suppressor': '#8C5593',
                        'Spliceosome': '#B2426B',
                        'Cohesin': '#15938A',
                        'Cell signalling': '#0B447C',
                        'mCA': grey4}

In [ ]:
extra_color_classes = {'DNA methylation': ['dodgerblue', 'lightskyblue', '#0570b0', '#74a9cf', '#d0d1e6', '#7194B5', 'dodgerblue', 'lightskyblue', '#0570b0', '#74a9cf', '#d0d1e6', '#7194B5'],
                      'Chromatin modifiers': ['#fec44f', '#F29F33', '#FF9700', '#E5AC2C', '#fec44f', '#F29F33', '#FF9700', '#E5AC2C'],
                      'Transcription factors': ['#6CDD66' '#6EB796', '#6CDD66' '#6EB796'] ,
                      'Transcriptional corepressors': ['#0A6D1F', '#5F8E72', '#0A6D1F', '#5F8E72'] ,
                      'Tumour suppressor': ['#bcbddc', '#54278f', '#9242C6', '#7621AD', 'darkviolet', 'purple', '#bcbddc', '#54278f', '#9242C6', '#7621AD', 'darkviolet', 'purple'],
                      'Spliceosome': ['#fa9fb5', '#f768a1', '#980043', '#C10552', '#fa9fb5', '#f768a1', '#980043', '#C10552'],
                      'Cohesin': ['#0B897D', '#0B897D', '#0B897D'],
                      'Cell signalling': ['#06396D', '#22405E', '#06396D', '#22405E'],
                      'mCA': ['#637382'],
                      'NPM1': [c1]}

In [ ]:
mutation_class_colors = {'NPM1': c1,
                        'DNA methylation': '#5098BC',
                        'Chromatin modifiers': '#E57E38',
                        'Transcription factors':  '#54AF51',
                        'Transcriptional corepressors':  '#1E8739',
                        'Tumour suppressor': '#A460AD',
                        'Spliceosome': '#C94776',
                        'Cohesin': '#17AA9F',
                        'Cell signalling': '#195BA6',
                        'mCA': grey4}

In [ ]:
mutation_classes = {'NPM1': 'NPM1',
                   'DNMT3A': 'DNA methylation',
                   'TET2': 'DNA methylation',
                   'IDH1': 'DNA methylation',
                   'IDH2': 'DNA methylation',
                   'ASXL1': 'Chromatin modifiers',
                   'EZH2': 'Chromatin modifiers',
                   'RUNX1': 'Transcription factors',
                   'CEBPA': 'Transcription factors',
                   'GATA2': 'Transcription factors',
                   'BCOR': 'Transcriptional corepressors',
                   'BCORL1': 'Transcriptional corepressors',
                   'TP53': 'Tumour suppressor',
                   'PPM1D': 'Tumour suppressor',
                   'CHEK2': 'Tumour suppressor',
                   'WT1': 'Tumour suppressor',
                   'CBL': 'Tumour suppressor',
                   'DDX41': 'Tumour suppressor',
                   'SRSF2': 'Spliceosome',
                   'SF3B1': 'Spliceosome',
                   'U2AF1': 'Spliceosome',
                   'ZRSR2': 'Spliceosome',
                   'RAD21': 'Cohesin',
                   'STAG2': 'Cohesin',
                   'FLT3': 'Cell signalling',
                   'KIT': 'Cell signalling',
                   'JAK2': 'Cell signalling',
                   'KRAS': 'Cell signalling',
                   'NRAS': 'Cell signalling',
                   'PTPN11': 'Cell signalling',
                   'CSF3R': 'Cell signalling',
                   'GNB1': 'Cell signalling',
                   'GNAS': 'Cell signalling',
                   'MPL': 'Cell signalling',
                    'mCA': 'mCA',
                   '19q CNLOH': 'mCA',
                   '15q CNLOH': 'mCA',
                   '4q CNLOH': 'mCA',
                   'X GAIN': 'mCA',
                   '19p CNLOH': 'mCA',
                   '9p CNLOH': 'mCA',
                   '7q LOSS': 'mCA',
                    'chrX': 'mCA',
                    'chr19p': 'mCA',
                    'chr9p': 'mCA',
                    'chr7q': 'mCA',
                    'chr4q': 'mCA',
                    'chr15q': 'mCA',
                    'chr19q': 'mCA',
                    'chr9': 'mCA',
                    'chr19': 'mCA',
                    'chr7': 'mCA',
                    'chr4': 'mCA',
                    'chr15': 'mCA'}

# Create dictionary of sample details etc.

In [ ]:
#create a dictionary of the sample details
cases = {} #e.g. {'C92_002': ['C92_002_s1', 'C92_002_s2', 'C92_002_s3', 'C92_002_s4'....]}
controls = {} #e.g. {'CNTRL_001': ['CNTRL_001_s1', 'CNTRL_001_s10', 'CNTRL_001_s2'...]}
cases_and_controls = {}
sample_ages = {} #e.g. {'C92_002_s1': 73..., 'C92_002_s2': 75.., 'C92_002_s3': 75.....}
sample_diagnosis_age = {} #e.g. {'C92_002': 81.,,, 'C92_003': 75.., 'C92_005': 70.....}
sample_DNA_amount = {} #e.g. {'C92_002_s1': '45', 'C92_002_s2': '50', 'C92_002_s3': '50'...}
matched_sample = {} #e.g. {'C92_002': 'CNTRL_169', 'C92_003': 'CNTRL_002'...}

with open('Data_files/UKCTOCS_samples_processed_information.csv') as csvfile:
    readreader = csv.reader(csvfile)
    row_count=0
    for row in readreader:
        if row_count>0:
            sample_name = row[1].split('_')[0]+'_'+row[1].split('_')[1]
            timepoint = row[1]
            sample_ages[timepoint]=float(row[8])
            sample_DNA_amount[timepoint]=row[4]
            if row[9]!='':
                if '_' in row[9]:
                    matched_sample_name = row[9].split('_')[0]+'_'+row[9].split('_')[1]
                    matched_sample[sample_name]=matched_sample_name
                    
            if sample_name in cases_and_controls.keys():
                cases_and_controls[sample_name].append(timepoint)
            else:
                cases_and_controls[sample_name]=[timepoint]
                    
            if row[0]=='Case':
                if sample_name in cases.keys():
                    cases[sample_name].append(timepoint)
                else:
                    cases[sample_name]=[timepoint]
                    
                sample_diagnosis_age[sample_name]=float(row[7])
                
            if row[0]=='Control':
                if sample_name in controls.keys():
                    controls[sample_name].append(timepoint)
                else:
                    controls[sample_name]=[timepoint]
                if sample_name in matched_sample.keys():
                    sample_diagnosis_age[sample_name]=sample_diagnosis_age[matched_sample[sample_name]]

        row_count+=1
        
cases_and_controls_sorted = {}
for k, v in cases_and_controls.items():
    ages_sorted = []
    for i in v:
        timepoint_number = int(i.split('_')[2][1:])
        ages_sorted.append((timepoint_number, i))
    sorted_v = sorted(ages_sorted, reverse = False)
    cases_and_controls_sorted[k]=[]
    for i in sorted_v:
        cases_and_controls_sorted[k].append(i[1])
        
# cases_and_controls_sorted

# Indel calling from the VarDictJava output

The pipeline's VarDictJava output (`*_VarDictJava_annotated.txt`) holds every call it makes,
SNVs included. The indel call set used below is produced from it here, applying the filters
described in the Supplementary Methods:

- indels only (SNVs are called by the beta-binomial error model instead);
- protein-changing (`AA_change` is not `.`);
- mean position in read > 8, so not confined to read ends;
- strand-bias Fisher phred < 60;
- ExAC allele frequency < 1%;
- at least 4 supporting reads, **unless** the variant is in *NPM1* or has been reported at
  least once in haematopoietic and lymphoid tissue in COSMIC v92.

This is run on the standard VarDictJava output.

In [ ]:
# Indel calls: filter the pipeline's annotated VarDictJava output down to the calls used here.
#
# The filters are those in the Supplementary Methods, applied per timepoint, followed by a
# cross-timepoint rescue: an indel is kept at a timepoint if it clears the read threshold at
# any timepoint in that individual.  This mirrors the rescue applied to the beta-binomial SNV
# calls, and matters because a clone that is unambiguous at one timepoint is often supported
# by only 2-3 reads at the timepoints where it is emerging or receding.  The rescue is
# deliberately permissive; the trajectory-level filters below decide what is real.

def replace_exac_0(exac):
    if exac == '.':
        new_exac = 0
    else:
        new_exac = exac
    return new_exac

def indel_calls_from_annotated(annotated_file):
    """Apply the per-timepoint indel filters to one annotated VarDictJava file."""
    df = pd.read_csv(annotated_file, sep='\t', low_memory=False)
    df = df[df['variant_type'] != 'SNV']                                  # indels only
    df = df[df['AA_change'] != '.']                                       # protein-changing
    df = df[df['mean_position_in_read'] > 8]                              # not only at read ends
    df = df[df['strand_bias_fisher_p_value_phred'] < 60]                  # not strand biased
    df['exac_all'] = df['exac_all'].apply(replace_exac_0).astype(float)
    df = df[df['exac_all'] < 0.01]                                        # not a common population variant
    df['variant_name'] = df['gene'] + ' ' + df['AA_change']
    df['variant_depth'] = df['variant_depth'].astype(float)
    df['cosmic_haem_lymphoid'] = df['cosmic_haem_lymphoid'].astype(float)
    return df

def passes_read_threshold(df):
    """>= 4 supporting reads, unless in NPM1 or seen in COSMIC haematopoietic/lymphoid tissue."""
    return (df['variant_depth'] > 3) | (df['gene'] == 'NPM1') | (df['cosmic_haem_lymphoid'] > 0)

def variant_positions(df):
    return set(zip(df['chromosome'], df['start'], df['REF'], df['ALT']))

annotated_by_individual = {}
for annotated_file in sorted(os.listdir(ANNOTATED_INDEL_DIR)):
    if not annotated_file.endswith('_DCS_VarDictJava_annotated.txt'):
        continue
    timepoint = re.match(r'((?:C92|CNTRL)_\d+_s\d+)', annotated_file).group(1)
    individual = timepoint.rsplit('_s', 1)[0]
    annotated_by_individual.setdefault(individual, []).append(annotated_file)

files_written = []
for individual, annotated_files in annotated_by_individual.items():
    filtered = {}
    detected_in_individual = set()
    for annotated_file in annotated_files:
        df = indel_calls_from_annotated(ANNOTATED_INDEL_DIR + annotated_file)
        filtered[annotated_file] = df
        detected_in_individual |= variant_positions(df[passes_read_threshold(df)])

    for annotated_file, df in filtered.items():
        rescued = [position in detected_in_individual
                   for position in zip(df['chromosome'], df['start'], df['REF'], df['ALT'])]
        df = df[passes_read_threshold(df) | pd.Series(rescued, index=df.index)]
        outfile = INDEL_DIR + annotated_file.replace('annotated.txt', 'indel_variant_calls.txt')
        df.to_csv(outfile, sep='\t', index=False)
        files_written.append(outfile)

print(str(len(files_written)) + ' indel call files written')


# Retrieve data files

In [ ]:
#Filenames for each timepoint for each sample (sampleS)...
n = 0
sample_filenames = {}
timepoints_available = {}

for sample in cases_and_controls_sorted.keys():
    filenames = []
    timepoints = cases_and_controls_sorted[sample]
    for time in timepoints:
        age = sample_ages[time]
        filename = SNV_DIR+time+'_SNV_SNV_watson_code_DCS_MUFs_3_beta_binomial_SNV_all_variant_calls_Oct_2023.txt'
        filename2 = SNV_DIR+time+'_SNV_watson_code_DCS_MUFs_3_beta_binomial_SNV_all_variant_calls_Oct_2023.txt'
        
        if os.path.isfile(filename):
            filenames.append(filename)
            if sample in timepoints_available.keys():
                timepoints_available[sample].append((age, time))
            else:
                timepoints_available[sample]=[(age, time)]
        if os.path.isfile(filename2):
            filenames.append(filename2)
            if sample in timepoints_available.keys():
                timepoints_available[sample].append((age, time))
            else:
                timepoints_available[sample]=[(age, time)]
    sample_filenames[sample]=filenames
    n+=1
    
timepoints_available_sorted = {}
for k, v in timepoints_available.items():
    v = sorted(v, reverse = True)
    just_times = []
    for i in v:
        just_times.append(i[1])
    timepoints_available_sorted[k]=just_times
    
# sample_filenames

In [ ]:
#Indel filenames for each timepoint for each sample (sampleS)...
n = 0
sample_filenames_indels = {}
timepoints_available_indels = {}

for sample in cases_and_controls_sorted.keys():
    filenames = []
    timepoints = cases_and_controls_sorted[sample]
    for time in timepoints:
        age = sample_ages[time]

        filename = INDEL_DIR+time+'_SNV_SNV_watson_code_DCS_VarDictJava_indel_variant_calls.txt'
        filename2 = INDEL_DIR+time+'_SNV_watson_code_DCS_VarDictJava_indel_variant_calls.txt'
        
        if os.path.isfile(filename):
            filenames.append(filename)
            if sample in timepoints_available_indels.keys():
                timepoints_available_indels[sample].append((age, time))
            else:
                timepoints_available_indels[sample]=[(age, time)]
        if os.path.isfile(filename2):
            filenames.append(filename2)
            if sample in timepoints_available_indels.keys():
                timepoints_available_indels[sample].append((age, time))
            else:
                timepoints_available_indels[sample]=[(age, time)]
    sample_filenames_indels[sample]=filenames
    n+=1
    
timepoints_available_sorted_indels = {}
for k, v in timepoints_available_indels.items():
    v = sorted(v, reverse = True)
    just_times = []
    for i in v:
        just_times.append(i[1])
    timepoints_available_sorted_indels[k]=just_times
    
# sample_filenames_indels

In [ ]:
#my mCA filenames for each timepoint for each sample (sampleS)...
n = 0
sample_filenames_mCAs = {}
timepoints_available_mCAs = {}

for sample in cases_and_controls_sorted.keys():
    filenames = []
    timepoints = cases_and_controls_sorted[sample]

    filename = MCA_DIR+sample+'_mCA.txt'

    if os.path.isfile(filename):
        filenames.append(filename)

    sample_filenames_mCAs[sample]=filenames
    n+=1
    
# sample_filenames_mCAs

# Functions for retrieving/ filtering variants and filtering trajectories

In [ ]:
trajectory_p_value_threshold = 1e-10
max_exac = 0.001 #difference compared to v5
max_strand_bias = 60
min_pos_read = 8
single_timepoint_chance_threshold = 0.05 #for variants detected at one middle timepoint only

# Outputs are written in place: per-timepoint post-processed files beside their inputs,
# combined tables into Data_files/.  Set OUT_DIR to a folder name to divert every output
# there instead (useful when comparing a re-run against a previous one).
OUT_DIR = None
if OUT_DIR:
    os.makedirs(OUT_DIR, exist_ok=True)

# Columns produced by the error model: kept in the working dataframes, dropped from the released tables.
error_model_columns = ['fitting method', 'iteration called at', 'p-value', 'MLE call',
                       'position final error rate', 'position final delta', 'total variants called at position']

# Variant types that are mCAs (reported separately in Supplementary Table 7, produced by the mCA caller).
mCA_variant_types = ['CNLOH', 'GAIN', 'LOSS']

# Column names used in the Supplementary Tables.
supplementary_table_column_names = {'sample name': 'Sample name', 'age_sample_taken': 'Age sample taken',
    'months_to_diagnosis': 'Months to diagnosis (in pre-AML case/ matched pre-AML case for controls)',
    'age_at_AML_diagnosis': 'Age at AML diagnosis (in pre-AML case/ matched pre-AML case in controls)',
    'matched_sample': 'Matched sample', 'chromosome': 'Chromosome', 'start': 'Start', 'end': 'End',
    'total_depth': 'Total depth', 'variant_depth': 'Variant depth', 'VAF (cell fraction for mCAs)': 'VAF',
    'intronic_exonic': 'Intronic or exonic', 'variant_type': 'Variant type', 'gene': 'Gene',
    'transcript': 'Transcript', 'exon': 'Exon', 'AA_change': 'Amino acid change',
    'exonic_function': 'Exonic function', 'cosmic_ID': 'COSMIC ID', 'cosmic_total': 'COSMIC total count',
    'cosmic_haem_lymphoid': 'COSMIC haem/lymphoid count', 'cosmic_sites': 'COSMIC sites',
    'exac_all': 'ExAC frequency (all)', 'gnomad_all': 'gnomAD frequency (all)'}


In [ ]:
def VAF_error_rate(variant_depth, total_depth, VAF):
    error = (np.sqrt(variant_depth*(1-VAF)))/total_depth
    return error

In [ ]:
def sample_variant_dictionaries(sample_name):
    sample_trajectories = {}

    sample_ages_processed = []
    for file in sample_filenames[sample_name]:
        sample = os.path.basename(file).split('_')[0]+'_'+os.path.basename(file).split('_')[1]+'_'+os.path.basename(file).split('_')[2]
        age = sample_ages[sample]
        sample_ages_processed.append(age)

    sample_ages_sorted = sorted(sample_ages_processed)

    ### CREATE DICTIONARIES OF ALL THE SAMPLE TRAJECTORIES (INCLUDING VARIANTS CALLED AS ERROR AND VARIANTS CALLED AS REAL) ###
    #SNVs
    for file in sample_filenames[sample_name]:
        sample = os.path.basename(file).split('_')[0]+'_'+os.path.basename(file).split('_')[1]+'_'+os.path.basename(file).split('_')[2]
        age = sample_ages[sample]
        df = pd.read_csv(file, sep = '\t')
        df_filtered = df[['position ID', 'p-value', 'call', 'total depth', 'variant depth',
                           'VAF', 'cosmic_haem_lymphoid', 'gene', 'AA_change']].copy()
        df_filtered['p-value'] = df_filtered['p-value'].replace(np.nan, 0) #convert np.nan p-values (if VAF >0.1) to 0
        df_dict = pd.DataFrame.to_dict(df_filtered, orient = 'index')
        for line, details in df_dict.items():
            p_value = details['p-value']
            position_ID = details['position ID']
            COSMIC_haem = details['cosmic_haem_lymphoid']
            VAF = details['VAF']
            variant_depth = details['variant depth']
            total_depth = details['total depth']
            VAF_error = VAF_error_rate(variant_depth, total_depth, VAF)
            call = details['call']
            variant = details['gene']+' '+details['AA_change']
            if (position_ID, variant, COSMIC_haem) in sample_trajectories.keys():
                sample_trajectories[(position_ID, variant, COSMIC_haem)][age]=(VAF, VAF_error, p_value, call, 'SNV') #call = REAL VARIANT or ERROR
            else:
                sample_trajectories[(position_ID, variant, COSMIC_haem)]= {}
                sample_trajectories[(position_ID, variant, COSMIC_haem)][age]=(VAF, VAF_error, p_value, call, 'SNV')

    #indels
    for file in sample_filenames_indels[sample_name]:
#         print(file)
        sample = os.path.basename(file).split('_')[0]+'_'+os.path.basename(file).split('_')[1]+'_'+os.path.basename(file).split('_')[2]
#         print(sample)
        age = sample_ages[sample]
        df = pd.read_csv(file, sep = '\t')
        df_filtered = df[['chromosome', 'start', 'REF', 'ALT', 'total_depth', 'variant_depth',
                           'VAF', 'cosmic_haem_lymphoid', 'gene', 'AA_change']]
        df_dict = pd.DataFrame.to_dict(df_filtered, orient = 'index')
        for line, details in df_dict.items():
            p_value = 0
            position_ID = details['chromosome']+','+str(details['start'])+','+details['REF']+','+details['ALT']
            COSMIC_haem = details['cosmic_haem_lymphoid']
            VAF = details['VAF']
            variant_depth = details['variant_depth']
            total_depth = details['total_depth']
            VAF_error = VAF_error_rate(variant_depth, total_depth, VAF)
            variant = details['gene']+' '+details['AA_change']
            if (position_ID, variant, COSMIC_haem) in sample_trajectories.keys():
                sample_trajectories[(position_ID, variant, COSMIC_haem)][age]=(VAF, VAF_error, p_value, 'REAL VARIANT', 'indel') #call = REAL VARIANT or ERROR
            else:
                sample_trajectories[(position_ID, variant, COSMIC_haem)]= {}
                sample_trajectories[(position_ID, variant, COSMIC_haem)][age]=(VAF, VAF_error, p_value, 'REAL VARIANT', 'indel')


    #mCAs
    if len(sample_filenames_mCAs[sample_name])>0:
        file = sample_filenames_mCAs[sample_name][0]
        with open(file) as csvfile:
            readreader = csv.reader(csvfile, delimiter = '\t')
            row_count=0
            for row in readreader:
                if row_count>0:
                    timepoint = row[0]
                    age = sample_ages[timepoint]
                    chromosome = row[1]
                    mCA = row[2]
                    cell_fraction = row[3]
                    if cell_fraction not in ['not processed', 'contaminated?']:
                        cell_fraction = float(cell_fraction)
                    if (mCA, '') in sample_trajectories.keys():
                        sample_trajectories[(mCA, '', '')][age]=(cell_fraction, 0, 0, 'REAL VARIANT', 'mCA')
                    if (mCA, '') not in sample_trajectories.keys():
                        sample_trajectories[(mCA, '', '')]= {}
                        sample_trajectories[(mCA, '', '')][age]=(cell_fraction, 0, 0, 'REAL VARIANT', 'mCA')

                row_count+=1

    ### RECORD 0 IF THE VARIANT WASN'T DETECTED AT A PARTICULAR TIMEPOINT ###
    all_sample_trajectories = {}
    for k, v in sample_trajectories.items():
        variant_and_cosmic = k
        updated_variant_dict = {}
        for i in sample_ages_sorted:
            if i in v.keys(): #i.e. if the variant was detected as that age
                updated_variant_dict[i]=v[i] 
            else: #i.e. if the variant was not detected at an age that was sampled
                updated_variant_dict[i]=(0, 0, np.nan, 'NOT DETECTED', '-')
        all_sample_trajectories[k]=updated_variant_dict
        
    VAF_ages_error_list = {}
    for k, v in sample_trajectories.items(): #sample trajectories = (position ID, COSMIC): {age: (VAF, VAF_error, p_value, 'REAL VARIANT', 'indel'), age: ....}
        results_list = []
        for age, results in v.items():
            results_list.append((age, results[0], results[1], results[2], results[3], results[4])) #age, VAF, VAF_error, p-value, call, SNV/indel/mCA
        VAF_ages_error_list[k]=results_list

    return VAF_ages_error_list

## STEP 1: RESCUE VARIANTS THAT WERE CALLED AS ERRORS IF P-VALUE <0.1 AND CALLED AS REAL WITH P-VALUE <1e-10 IN ANY OF THE OTHER TIMEPOINTS IN THE TRAJECTORY

### Create a dictionary of ALL variants and their trajectories (errors + real variants)

In [ ]:
#create a dictionary where the key is the sample name and the values is a dictionary of the variant and the trajectory information
sample_trajectories = {}
for sample in cases_and_controls.keys():
    trajectory = sample_variant_dictionaries(sample)
    sample_trajectories[sample]=trajectory
    
# sample_trajectories

### Create a dictionary of just the variants that are called as real (with p-value <1e-10) at 1 or more timepoints

In [ ]:
sample_trajectories_p_value_filtered = {}

for sample, variants in sample_trajectories.items(): # e.g. C92_002: {(position_ID, COSMIC): [(age, VAF, VAF_error, p-value, call, variant_type), (age, VAF, VAF_error, p-value, call, variant_type) etc...]}
    real_variants = []
    for variant, trajectory in variants.items(): #(position_ID, COSMIC): [(age, VAF, VAF_error, p-value, call, variant_type), (age, VAF, VAF_error, p-value, call, variant_type) etc...]
        for i in trajectory:
            COSMIC_freq = variant[2]
            if COSMIC_freq == '':
                COSMIC_freq = 0
            age = i[0]
            VAF = i[1]
            VAF_error = i[2]
            p_value = i[3]
            call = i[4]
            variant_type = i[5]
            if call == 'REAL VARIANT':
                if float(COSMIC_freq) >= 10: #call as real if 'REAL VARIANT' and in COSMIC >=10 times
                    real_variants.append(variant)
                else:
                    if p_value <= trajectory_p_value_threshold: #if not in COSMIC it needs to have a p-value < given threshold (e.g. 1e-10) to be called as real
                        real_variants.append(variant) #create a list of all variants that were called as real at one or more timepoint in a trajectory

    #now call all variants as real if it was called as real at any other timepoint in it's trajectory (with p-value <1e-10) AND it's p-value at this timepoint of interest is <0.1            
    real_variants_in_trajectory={}
    for variant, trajectory in variants.items():
        if variant in real_variants: #if any of the variants in the trajectory were called as real at any timepoint...
            for i in trajectory:
                age = i[0]
                VAF = i[1]
                VAF_error = i[2]
                p_value = i[3]
                call = i[4]
                variant_type = i[5]
                if p_value < 0.1:
                    if variant in real_variants_in_trajectory.keys():
                        real_variants_in_trajectory[variant].append((age, VAF, VAF_error, p_value, call, variant_type)) #create a list of all variants that were called as real in that trajectory
                    else:
                        real_variants_in_trajectory[variant]=[(age, VAF, VAF_error, p_value, call, variant_type)] #create a list of all variants that were called as real in that trajectory
            
    if len(real_variants_in_trajectory)>0: #i.e. if there are real variants in the trajectory:
        sample_trajectories_p_value_filtered[sample]=real_variants_in_trajectory
            
# sample_trajectories_p_value_filtered

## STEP 2: FILTER OUT OVERLY COMMON VARIANTS:

### Non-germline variants

In [ ]:
def check_for_highly_common_variants_SNV_or_indels(cases_and_controls, sample_trajectories_p_value_filtered, SNV_or_indel): #SNVs
    
    #retrieve the variants
    variants_numbers = {}
    variants_numbers_cases = {}
    variants_numbers_controls = {}
    
    total_people = 0
    total_cases = 0
    total_controls = 0
    
    for sample_name in cases_and_controls:
        total_people+=1
        samples_variants = {}
        
        if 'C92'in sample_name:
            total_cases+=1
        if 'CNTRL' in sample_name:
            total_controls +=1
    
        samples_variants = sample_trajectories_p_value_filtered[sample_name] #will generate a dictionary of variants: trajectories in that person
        
        for variant, trajectory in samples_variants.items():
            variant_type = trajectory[0][5]
            if variant_type == SNV_or_indel:
                total_VAF = 0
                number_timepoints = len(trajectory)
                for i in trajectory:
                    total_VAF+=i[1]

                if total_VAF/number_timepoints <0.4: #i.e. try not to exclude germline variants at this stage:
                    if variant in variants_numbers.keys():
                        variants_numbers[variant]+=1
                    else:
                        variants_numbers[variant]=1

                    if 'C92'in sample_name:
                        if variant in variants_numbers_cases.keys():
                            variants_numbers_cases[variant]+=1
                        else:
                            variants_numbers_cases[variant]=1

                    if 'CNTRL'in sample_name:
                        if variant in variants_numbers_controls.keys():
                            variants_numbers_controls[variant]+=1
                        else:
                            variants_numbers_controls[variant]=1
            
    # all variants        
    variant_prevalence = {}
    for k, v in variants_numbers.items():
        prevalence = v/total_people
        variant_prevalence[k]=(v, prevalence)
        
    variant_prevalence_cases = {}
    for k, v in variants_numbers_cases.items():
        prevalence = v/total_cases
        variant_prevalence_cases[k]=(v, prevalence)
        
    variant_prevalence_controls = {}
    for k, v in variants_numbers_controls.items():
        prevalence = v/total_controls
        variant_prevalence_controls[k]=(v, prevalence)
                
    all_variants = {'cases and controls': variant_prevalence, 'cases': variant_prevalence_cases, 'controls':variant_prevalence_controls}
                
    return all_variants

In [ ]:
# Create lists of variants to exclude
cases_and_controls_list = list(cases.keys())+list(controls.keys())
variant_prevalence = check_for_highly_common_variants_SNV_or_indels(cases_and_controls_list, sample_trajectories_p_value_filtered, 'SNV')
indels_prevalence = check_for_highly_common_variants_SNV_or_indels(cases_and_controls_list, sample_trajectories_p_value_filtered, 'indel')

variants_to_exclude = {}
for k, v in variant_prevalence['controls'].items():
    if v[1]>=0.05: #if observed in >=5% of cases + controls
        if k[2]==0: #i.e. if nunber of observations in COSMIC = 0
            variants_to_exclude[k]=v
                                   
indels_to_exclude = {}
for k, v in indels_prevalence['controls'].items():
    if v[1]>=0.05: #if observed in >=5% of cases + controls
        if 'NPM1' not in k[1]:
            if k[2]==0: #i.e. if nunber of observations in COSMIC = 0
                indels_to_exclude[k]=v
                
# print('SNVs to exclude:')
# for k, v in variants_to_exclude.items():
#     print(k, v)
    
# print()
# print('indels to exclude:')
# for k, v in indels_to_exclude.items():
#     print(k, v)

In [ ]:
# create a list of the variants and trajectory timepoints that have already been inferred to be real (pre- common COSMIC filtering)
sample_trajectories_p_value_filtered_common_removed={}

# for sample, variants in sample_trajectories_p_value_filtered_common_variants.items():
for sample, variants in sample_trajectories_p_value_filtered.items():
    sample_variant_timepoints={}
    for variant_cosmic, trajectory in variants.items():
        if variant_cosmic not in variants_to_exclude.keys():
            if variant_cosmic not in indels_to_exclude.keys():
                sample_variant_timepoints[variant_cosmic]=trajectory
    if len(sample_variant_timepoints)>0:
        sample_trajectories_p_value_filtered_common_removed[sample]=sample_variant_timepoints

### Germline variants

In [ ]:
def check_for_highly_common_germline_variants(cases_and_controls, sample_trajectories_p_value_filtered, SNV_or_indel): #SNVs
    
    #retrieve the variants
    variants_numbers = {}
    variants_numbers_cases = {}
    variants_numbers_controls = {}
    
    total_people = 0
    total_cases = 0
    total_controls = 0
    
    for sample_name in cases_and_controls:
        total_people+=1
        samples_variants = {}
        
        if 'C92'in sample_name:
            total_cases+=1
        if 'CNTRL' in sample_name:
            total_controls +=1
    
        samples_variants = sample_trajectories_p_value_filtered[sample_name] #will generate a dictionary of variants: trajectories in that person
        
        for variant, trajectory in samples_variants.items():
            variant_type = trajectory[0][5]
            if variant_type == SNV_or_indel:
                total_VAF = 0
                number_timepoints = len(trajectory)
                for i in trajectory:
                    total_VAF+=i[1]

                if total_VAF/number_timepoints >=0.4: #i.e. try not just look at germline variants:
                    if variant in variants_numbers.keys():
                        variants_numbers[variant]+=1
                    else:
                        variants_numbers[variant]=1

                    if 'C92'in sample_name:
                        if variant in variants_numbers_cases.keys():
                            variants_numbers_cases[variant]+=1
                        else:
                            variants_numbers_cases[variant]=1

                    if 'CNTRL'in sample_name:
                        if variant in variants_numbers_controls.keys():
                            variants_numbers_controls[variant]+=1
                        else:
                            variants_numbers_controls[variant]=1
            
    # all variants        
    variant_prevalence = {}
    for k, v in variants_numbers.items():
        prevalence = v/total_people
        variant_prevalence[k]=(v, prevalence)
        
    variant_prevalence_cases = {}
    for k, v in variants_numbers_cases.items():
        prevalence = v/total_cases
        variant_prevalence_cases[k]=(v, prevalence)
        
    variant_prevalence_controls = {}
    for k, v in variants_numbers_controls.items():
        prevalence = v/total_controls
        variant_prevalence_controls[k]=(v, prevalence)
                
    all_variants = {'cases and controls': variant_prevalence, 'cases': variant_prevalence_cases, 'controls':variant_prevalence_controls}
                
    return all_variants

In [ ]:
# Create lists of variants to exclude
cases_and_controls_list = list(cases.keys())+list(controls.keys())
germline_variant_prevalence = check_for_highly_common_germline_variants(cases_and_controls_list, sample_trajectories_p_value_filtered, 'SNV')
germline_indels_prevalence = check_for_highly_common_germline_variants(cases_and_controls_list, sample_trajectories_p_value_filtered, 'indel')

germline_variants_to_exclude = {}
for k, v in germline_variant_prevalence['controls'].items():
    if v[1]>=0.05: #if observed in >=5% of cases + controls
        if k[2]<=5: #i.e. if nunber of observations in COSMIC = 0
            germline_variants_to_exclude[(k[0], k[1])]=v
                                   
germline_indels_to_exclude = {}
for k, v in indels_prevalence['controls'].items():
    if v[1]>=0.05: #if observed in >=5% of cases + controls
        if k[2]<=5: #i.e. if nunber of observations in COSMIC = 0
            germline_indels_to_exclude[(k[0], k[1])]=v
                
# print('SNVs to exclude:')
# for k, v in germline_variants_to_exclude.items():
#     print(k, v)
    
# print()
# print('indels to exclude:')
# for k, v in germline_indels_to_exclude.items():
#     print(k, v)

## STEP 2: FILTER OUT LIKELY ERROR TRAJECTORIES

In [ ]:
def extract_depth(info):
    depth = int(info.split(';')[2].split('=')[1])
    return depth

In [ ]:
def mean_depth_across_sample(sample_name, sample_timepoint):
    try:
        df= pd.read_csv(VCF_DIR+sample_timepoint+'_SNV_SNV_watson_code_DCS_variants_MUFs_3_all_positions.vcf', comment = '#', sep = '\t', header = None, names = ['chromosome', 'position', 'ID', 'REF','ALT', 'FILTER', 'INFO', 'FORMAT', 'SAMPLE'])
    except FileNotFoundError:
        df= pd.read_csv(VCF_DIR+sample_timepoint+'_SNV_watson_code_DCS_variants_MUFs_3_all_positions.vcf', comment = '#', sep = '\t', header = None, names = ['chromosome', 'position', 'ID', 'REF','ALT', 'FILTER', 'INFO', 'FORMAT', 'SAMPLE'])
    df['depth'] = df['INFO'].apply(extract_depth)
    depth_list = df['depth'].tolist()
    mean_depth = np.mean(depth_list)
    std_depths = np.std(depth_list)
    return mean_depth, std_depths

In [ ]:
def retrieve_depth_position(sample_name, sample_timepoint, chromosome, position): #get the total read depth at a position from the all positions vcf file
    try:
        df= pd.read_csv(VCF_DIR+sample_timepoint+'_SNV_SNV_watson_code_DCS_variants_MUFs_3_all_positions.vcf', comment = '#', sep = '\t', header = None, names = ['chromosome', 'position', 'ID', 'REF','ALT', 'FILTER', 'INFO', 'FORMAT', 'SAMPLE'])
    except FileNotFoundError:
        df= pd.read_csv(VCF_DIR+sample_timepoint+'_SNV_watson_code_DCS_variants_MUFs_3_all_positions.vcf', comment = '#', sep = '\t', header = None, names = ['chromosome', 'position', 'ID', 'REF','ALT', 'FILTER', 'INFO', 'FORMAT', 'SAMPLE'])
    df = df[(df['chromosome']==chromosome) & (df['position']==position)]
    df['depth'] = df['INFO'].apply(extract_depth)
    total_depth = df['depth'].tolist()
    return total_depth[0]

In [ ]:
def retrieve_sample_name_at_age(age, sample, sample_ages): #e.g. find out the timepoint name of the sample (e.g. C92_007_s2) if you know the same name (C92_007) and the age of the person
    for k, v in sample_ages.items():
        sample_abbrev= k.split('_')[0]+'_'+k.split('_')[1]
        if sample_abbrev == sample:
            if float(v) == float(age):
                timepoint_name = k
    return timepoint_name

In [ ]:
def growth_rate_calculation(VAF1, VAF3, age1, age3):
    return (np.log(VAF3/VAF1))/(age3-age1)

In [ ]:
def expected_VAF_and_reads(starting_VAF, growth_rate_per_year, time_interval, total_depth):
    s = growth_rate_per_year
    t = time_interval
    expected_VAF = starting_VAF*(np.exp(s*t))
#     expected_VAF = starting_VAF*((np.exp(s*t)-1)/s)
    expected_reads = expected_VAF*total_depth
    return expected_VAF, expected_reads

#### SNVs

In [ ]:
def filter_variants_and_trajectories_with_holes(sample_name, max_exac, max_strand_bias, min_pos_read, sample_filenames, trajectory_p_value_threshold, sample_variant_timepoints):
        
    #retrieve the variants
    samples_variants = {}
    final_timepoint_variants = []
    ages_with_samples_processed = []
    # print('SNVs:')
    # print()
    
    for file in sample_filenames[sample_name]:
        sample = os.path.basename(file).split('_')[0]+'_'+os.path.basename(file).split('_')[1]+'_'+os.path.basename(file).split('_')[2]
        mean_depth_across_panel, std_across_panel = mean_depth_across_sample(sample_name, sample)
        if mean_depth_across_panel >=100:
            # print('mean depth across panel = >100 for '+sample)
            age = sample_ages[sample]
            ages_with_samples_processed.append(age)
            df = pd.read_csv(file, sep = '\t')
            
            df_filtered = df[['position ID', 'p-value', 'total depth', 'variant depth', 'VAF', 'cosmic_haem_lymphoid',  'RSID',
                              'gene', 'AA_change', 'exonic_function', 'strand_bias_fisher_p_value_phred', 'mean_position_in_read', 'exac_all']].copy()
            df_filtered['p-value'] = df_filtered['p-value'].replace(np.nan, 0) #convert np.nan p-values (if VAF >0.1) to 0
            df_dict = pd.DataFrame.to_dict(df_filtered, orient = 'index')
            for line, details in df_dict.items():
                position_ID = details['position ID']
                chromosome = position_ID.split(',')[0]
                position = position_ID.split(',')[1]
                ref = position_ID.split(',')[2]
                alt = position_ID.split(',')[3]
                p_value = details['p-value']
                COSMIC_haem = details['cosmic_haem_lymphoid']
                VAF = details['VAF']
                variant_depth = details['variant depth']
                total_depth = details['total depth']
                error_rate = VAF_error_rate(variant_depth, total_depth, VAF)
                AA_change = details['AA_change']
                gene = details['gene']
                variant = gene+' '+AA_change
                variant_type = details['exonic_function'] #e.g. synonymous or non-synonymous
                mean_pos_read = details['mean_position_in_read']
                if mean_pos_read == '.':
                    mean_pos_read = 100
                exac_all = details['exac_all']
                if exac_all == '.':
                    exac_all = 0
                strand_bias = float(details['strand_bias_fisher_p_value_phred'])
                RSID = details['RSID']

                if (position_ID, variant) in sample_variant_timepoints[sample_name]:
                    if age in sample_variant_timepoints[sample_name][(position_ID, variant)]: #make sure the variant has been called as real in previous filter step
                        # print(variant)
                        # print('age '+str(age)+' in sample_variant_timepoints')
                        if AA_change != '.':
#                             if RSID == '-':
#                                 if float(exac_all) < max_exac:
                            if strand_bias < max_strand_bias:
                                # print('strand bias < max strand bias')
                                if float(mean_pos_read) > min_pos_read:
                                    # print('mean pos read > min pos read')
                                    if variant_type.split(' ')[0]!= 'synonymous':
                                        if total_depth >= 500:
                                            if gene!='FLT3':
                                                if variant_depth >1:
                                                    if (position_ID, variant) in samples_variants.keys():
                                                        samples_variants[(position_ID, variant)].append((age, VAF, p_value, variant_depth, total_depth, (chromosome, position, ref, alt), error_rate, file, COSMIC_haem, exac_all, RSID))
                                                    else:
                                                        samples_variants[(position_ID, variant)]=[(age, VAF, p_value, variant_depth, total_depth, (chromosome, position, ref, alt), error_rate, file, COSMIC_haem, exac_all, RSID)] 
                                            if gene == 'FLT3':
                                                if variant_type == 'internal tandem duplication':
                                                    if (position_ID, variant) in samples_variants.keys():
                                                        samples_variants[(position_ID, variant)].append((age, VAF, p_value, variant_depth, total_depth, (chromosome, position, ref, alt), error_rate, file, COSMIC_haem, exac_all, RSID))
                                                    else:
                                                        samples_variants[(position_ID, variant)]=[(age, VAF, p_value, variant_depth, total_depth, (chromosome, position, ref, alt), error_rate, file, COSMIC_haem, exac_all, RSID)] 
                                                else:
                                                    if variant_depth >1:
                                                        if (position_ID, variant) in samples_variants.keys():
                                                            samples_variants[(position_ID, variant)].append((age, VAF, p_value, variant_depth, total_depth, (chromosome, position, ref, alt), error_rate, file, COSMIC_haem, exac_all, RSID))
                                                        else:
                                                            samples_variants[(position_ID, variant)]=[(age, VAF, p_value, variant_depth, total_depth, (chromosome, position, ref, alt), error_rate, file, COSMIC_haem, exac_all, RSID)] 
                                        else:
                                            if total_depth >= (float(mean_depth_across_panel)-(2*float(std_across_panel))):
                                                # print('2*std depth = '+str(2*float(std_across_panel))+' for variant '+variant)
                                                if gene!='FLT3':
                                                    if variant_depth >1:
                                                        # print('add variant to samples_variants')
                                                        if (position_ID, variant) in samples_variants.keys():
                                                            samples_variants[(position_ID, variant)].append((age, VAF, p_value, variant_depth, total_depth, (chromosome, position, ref, alt), error_rate, file, COSMIC_haem, exac_all, RSID))
                                                        else:
                                                            samples_variants[(position_ID, variant)]=[(age, VAF, p_value, variant_depth, total_depth, (chromosome, position, ref, alt), error_rate, file, COSMIC_haem, exac_all, RSID)] 
                                                if gene == 'FLT3':
                                                    if variant_type == 'internal tandem duplication':
                                                        if (position_ID, variant) in samples_variants.keys():
                                                            samples_variants[(position_ID, variant)].append((age, VAF, p_value, variant_depth, total_depth, (chromosome, position, ref, alt), error_rate, file, COSMIC_haem, exac_all, RSID))
                                                        else:
                                                            samples_variants[(position_ID, variant)]=[(age, VAF, p_value, variant_depth, total_depth, (chromosome, position, ref, alt), error_rate, file, COSMIC_haem, exac_all, RSID)] 
                                                    else:
                                                        if variant_depth >1:
                                                            if (position_ID, variant) in samples_variants.keys():
                                                                samples_variants[(position_ID, variant)].append((age, VAF, p_value, variant_depth, total_depth, (chromosome, position, ref, alt), error_rate, file, COSMIC_haem, exac_all, RSID))
                                                            else:
                                                                samples_variants[(position_ID, variant)]=[(age, VAF, p_value, variant_depth, total_depth, (chromosome, position, ref, alt), error_rate, file, COSMIC_haem, exac_all, RSID)] 
        #                                     else:
        #                                         print(variant+ 'total depth <500 and < mean-2xs.d.')

        # else:
        #     print('mean depth for '+sample+' <100 so filtered out')

    #sort results for each variant by age order
    samples_variants_sorted = {}
    for k, v in samples_variants.items():
        variant = k
        timepoints_list = v
        timepoints_list = sorted(timepoints_list)
        samples_variants_sorted[k]=timepoints_list
        
    ##### LOOK FOR TRAJECTORIES WITH GAPS IN THEM #######
    samples_variants_filtered = {}
    for k, v in samples_variants_sorted.items():
        list_of_detected_variants= v
        variant = k
        position_information = v[0][5]
        chromosome = position_information[0]
        position = int(position_information[1])
        ref = position_information[2]
        youngest_age_variant_detected = v[0][0]
        oldest_age_variant_detected = v[-1][0]

        #--- variants detected at a single, middle timepoint ------------------------
        # No growth rate can be estimated from one detection, but under any monotonic
        # trajectory the clone must have been at least as large as the observed VAF at
        # either the preceding or the following timepoint.  Taking the larger of the two
        # resulting probabilities gives a conservative bound on the chance of having
        # missed it, with no assumption about the growth rate.  The first and last
        # timepoints are exempt: a declining clone has no earlier constraint and an
        # emerging clone has no later constraint.
        if len(v) == 1:
            ages_in_order = sorted(ages_with_samples_processed)
            position_of_detection = ages_in_order.index(v[0][0])
            if 0 < position_of_detection < (len(ages_in_order)-1):
                VAF_detected = v[0][1]
                chance_either_side = []
                try:
                    for neighbouring_age in (ages_in_order[position_of_detection-1], ages_in_order[position_of_detection+1]):
                        neighbouring_sample = retrieve_sample_name_at_age(neighbouring_age, sample_name, sample_ages)
                        neighbouring_depth = retrieve_depth_position(sample_name, neighbouring_sample, chromosome, position)
                        chance_either_side.append(np.exp(-VAF_detected*float(neighbouring_depth)))
                except (IndexError, KeyError, FileNotFoundError, TypeError, UnboundLocalError):
                    chance_either_side = [] #if the depth cannot be retrieved, keep the variant
                if len(chance_either_side)==2 and max(chance_either_side) < single_timepoint_chance_threshold:
                    # print(k[1]+' excluded: detected only at a middle timepoint, chance of missing it either side = '+str(max(chance_either_side)))
                    continue


        oldest_age_position_in_ages_list = ages_with_samples_processed.index(oldest_age_variant_detected)
        youngest_age_position_in_ages_list = ages_with_samples_processed.index(youngest_age_variant_detected) #position of youngest age detected in ages_with_samples_processed list

        expected_number_of_ages = oldest_age_position_in_ages_list - youngest_age_position_in_ages_list+1
        number_of_ages_detected = len(v)
        total_ages_missing = expected_number_of_ages - number_of_ages_detected
        # print(variant)
        # print('expected number of ages = ', expected_number_of_ages)
        # print('number of ages detected = ', number_of_ages_detected)
        # print('total ages missing = ', total_ages_missing)

        proportion_of_ages_missing = total_ages_missing/expected_number_of_ages

        if total_ages_missing <0:
            samples_variants_filtered[k]=list_of_detected_variants

        else: #if there are timepoints missing when you might expect to see them...
            #look to see if should exclude the whole trajectory based on the number missing (if >1/3 missing and total chance of missing is <5%) (i.e. keep them if chance of missing is high (and they were missed))
            ages_missing = []
            ages_detected = [i[0] for i in v]
            for i in ages_with_samples_processed[youngest_age_position_in_ages_list: oldest_age_position_in_ages_list]:
                if i not in ages_detected:
                    ages_missing.append(i)

            # print('ages missing = ', ages_missing)

            excluded = 0
            if proportion_of_ages_missing >= 1/3:
                chance_of_missing_list = []
                for age in ages_missing:
                    sample_timepoint_name = retrieve_sample_name_at_age(age, sample_name, sample_ages) #sample_ages is a list of timepoint names and their ages
                    total_read_depth = retrieve_depth_position(sample_name, sample_timepoint_name, chromosome, position)  #read depth at that position in the missing sample

                    for i in ages_detected:
                        if i<age:
                            previous_age = i
                        if i>age:
                            next_age = i
                            break

                    previous_position_in_ages_detected_list = ages_detected.index(previous_age)
                    previous_VAF = v[previous_position_in_ages_detected_list][1]

                    next_position_in_ages_detected_list = ages_detected.index(next_age)
                    next_VAF = v[next_position_in_ages_detected_list][1]
                    
                    growth_rate = growth_rate_calculation(previous_VAF, next_VAF, previous_age, next_age)
                    years_since_previous_timepoint = age-previous_age

                    expected_VAF, expected_reads = expected_VAF_and_reads(previous_VAF, growth_rate, years_since_previous_timepoint, total_read_depth)
                    chance_of_missing = np.exp(-expected_reads)
                    chance_of_missing_list.append(chance_of_missing)
                    # print('previous VAF = ', previous_VAF)
                    # print('next VAF = ', next_VAF)
                    # print('expected VAF = ', expected_VAF)
                    # print('chance of missing = ', chance_of_missing)

                # print('chance of missing list', chance_of_missing_list)
                total_chance_of_missing = np.prod(chance_of_missing_list)
                # print('total chance of missing = ', total_chance_of_missing)

                if total_chance_of_missing <0.05:
                    # print(k[1]+' trajectory completely excluded becuase >=1/3 timepoints missing and chance of this is '+str(total_chance_of_missing))
                    excluded+=1 

            if excluded ==0: #i.e. whole trajectory hasn't been excluded, look to see if earlier timepoints should be
                list_of_detected_variants = list_of_detected_variants
                for age in sorted(ages_missing): #e.g. [70, 72]...
#                     print('age = ', age)
                    sample_timepoint_name = retrieve_sample_name_at_age(age, sample_name, sample_ages) #sample_ages is a list of timepoint names and their ages (sample_timepoint_name = e.g. C92_002_s7) = needed to retrieve depth at that position
                    total_read_depth = retrieve_depth_position(sample_name, sample_timepoint_name, chromosome, position)  #read depth at that position in the missing sample
#                     print('sample timepoint name =', sample_timepoint_name)

                    for i in ages_detected:
                        if i<age:
                            previous_age = i
                        if i>age:
                            next_age = i
                            break

                    if previous_age in ages_detected: #might not be there id the samples missing are 2 consecutive timepoints and it was removed in the previous iteration
                        previous_position_in_ages_detected_list = ages_detected.index(previous_age)
                        previous_VAF = v[previous_position_in_ages_detected_list][1]

                        next_position_in_ages_detected_list = ages_detected.index(next_age)
                        next_VAF = v[next_position_in_ages_detected_list][1]

                        growth_rate = growth_rate_calculation(previous_VAF, next_VAF, previous_age, next_age)
                        years_since_previous_timepoint = age-previous_age

                        expected_VAF, expected_reads = expected_VAF_and_reads(previous_VAF, growth_rate, years_since_previous_timepoint, total_read_depth)
                        chance_of_missing = np.exp(-expected_reads)
                        # print('total read depth = ', total_read_depth)
                        # print('chance of missing = ', chance_of_missing)

                        if chance_of_missing<0.05:
                            list_of_detected_variants = list_of_detected_variants[next_position_in_ages_detected_list:] #i.e. start the list at the position of the next VAF
                            ages_detected = ages_detected[next_position_in_ages_detected_list:] #start the list of ages detected at the next detected sample
                            # print('samples before '+sample_timepoint_name+' excluded due to low chance of missing '+sample_timepoint_name)

                if len(list_of_detected_variants)>0: #keep whatever is left of the list
                    samples_variants_filtered[k]=list_of_detected_variants
            
    return samples_variants_filtered, sorted(ages_with_samples_processed)

#### indels

In [ ]:
def COSMIC_frequency_indels(sample_name, sample_timepoint, gene, variant):
    try:
        df= pd.read_csv(INDEL_DIR+sample_timepoint+'_SNV_SNV_watson_code_DCS_VarDictJava_indel_variant_calls_Sept_2023.txt', sep = '\t')
    except FileNotFoundError:
        df= pd.read_csv(INDEL_DIR+sample_timepoint+'_SNV_watson_code_DCS_VarDictJava_indel_variant_calls_Sept_2023.txt', sep = '\t')
    df = df[(df['gene']==gene) & (df['AA_change']==variant)]
    cosmic = df['cosmic_haem_lymphoid'].tolist()
    
    if len(cosmic)==0:
        print(gene+'_'+variant+' not found in '+sample_timepoint+' vardict file')
        cosmic = [0]
    
    return cosmic[0]

In [ ]:
def filter_indels_and_trajectories_with_holes(sample_name, max_exac, max_strand_bias, min_pos_read, sample_filenames, sample_variant_timepoints):
        
    #retrieve the variants
    samples_variants = {}
    final_timepoint_variants = []
    ages_with_samples_processed = []
    # print()
    # print('INDELS:')
    # print()
    
    #indels
    for file in sample_filenames_indels[sample_name]:
        sample = os.path.basename(file).split('_')[0]+'_'+os.path.basename(file).split('_')[1]+'_'+os.path.basename(file).split('_')[2]
        mean_depth_across_panel, std_across_panel = mean_depth_across_sample(sample_name, sample)
        if mean_depth_across_panel <100:                                                             
            print('mean depth for '+sample+' <100 so filtered out')
            # print()
        if mean_depth_across_panel >=100:
            age = sample_ages[sample]
            ages_with_samples_processed.append(age)
            df = pd.read_csv(file, sep = '\t')
            df_filtered = df[['chromosome', 'start', 'REF', 'ALT', 'total_depth', 'variant_depth', 'VAF', 'cosmic_haem_lymphoid',
                              'gene', 'AA_change', 'exonic_function', 'strand_bias_fisher_p_value_phred', 'mean_position_in_read', 'exac_all']]
            df_dict = pd.DataFrame.to_dict(df_filtered, orient = 'index')
            
            for line, details in df_dict.items():
                position_ID = details['chromosome']+','+str(details['start'])+','+details['REF']+','+details['ALT']
                chromosome = details['chromosome']
                position = int(details['start'])
                ref = details['REF']
                alt = details['ALT']
                cosmic_frequency = details['cosmic_haem_lymphoid']
                VAF = details['VAF']
                variant_depth = details['variant_depth']
                total_depth = details['total_depth']
                error_rate = VAF_error_rate(variant_depth, total_depth, VAF)
                AA_change = details['AA_change']
                gene = details['gene']
                variant = gene+' '+AA_change
                variant_type = details['exonic_function'] #e.g. synonymous or non-synonymous
                mean_pos_read = details['mean_position_in_read']
                if mean_pos_read == '.':
                    mean_pos_read = 100
                exac_all = details['exac_all']
                if exac_all == '.':
                    exac_all = 0
                strand_bias = float(details['strand_bias_fisher_p_value_phred'])

                if (position_ID, variant) in sample_variant_timepoints[sample_name]:
#                     print(variant)
                    if age in sample_variant_timepoints[sample_name][(position_ID, variant)]: #make sure the variant has been called as real in previous filter step
                        if AA_change != '.':
    #                         if RSID == '-':
#                             if float(exac_all) < max_exac:
                            if strand_bias < max_strand_bias:
                                if float(mean_pos_read) > min_pos_read:
                                    if (gene == 'NPM1') and (variant_type.split(' ')[0]== 'frameshift'):#if none of the timepoints have VAF>4%, rescue them if NPM1 288 frameshift or if variant is in COSMIC haem 1 or more times
                                        amino_acid = int(AA_change.split('*')[0].split('fs')[0].split('.')[1][1:-1])
                                        if amino_acid == 288:
                                            if (position_ID, variant) in samples_variants.keys():
                                                samples_variants[(position_ID, variant)].append((age, VAF, error_rate, variant_depth, total_depth, (chromosome, position, ref, alt), file, cosmic_frequency, variant_type, exac_all))
                                            else:
                                                samples_variants[(position_ID, variant)]=[(age, VAF, error_rate, variant_depth, total_depth, (chromosome, position, ref, alt), file, cosmic_frequency, variant_type, exac_all)]
                                        else: #if NPM1 frameshift but not hotspot
                                            if total_depth >500:
                                                if (position_ID, variant) in samples_variants.keys():
                                                    samples_variants[(position_ID, variant)].append((age, VAF, error_rate, variant_depth, total_depth, (chromosome, position, ref, alt), file, cosmic_frequency, variant_type, exac_all))
                                                else:
                                                    samples_variants[(position_ID, variant)]=[(age, VAF, error_rate, variant_depth, total_depth, (chromosome, position, ref, alt), file, cosmic_frequency, variant_type, exac_all)]
                                            else:
                                                if total_depth >= (float(mean_depth_across_panel)-(2*float(std_across_panel))):
#                                                         if total_depth >=200:
                                                    if (position_ID, variant) in samples_variants.keys():
                                                        samples_variants[(position_ID, variant)].append((age, VAF, error_rate, variant_depth, total_depth, (chromosome, position, ref, alt), file, cosmic_frequency, variant_type, exac_all))
                                                    else:
                                                        samples_variants[(position_ID, variant)]=[(age, VAF, error_rate, variant_depth, total_depth, (chromosome, position, ref, alt), file, cosmic_frequency, variant_type, exac_all)]

                                    elif total_depth >500:
                                        if (position_ID, variant) in samples_variants.keys():
                                            samples_variants[(position_ID, variant)].append((age, VAF, error_rate, variant_depth, total_depth, (chromosome, position, ref, alt), file, cosmic_frequency, variant_type, exac_all))
                                        else:
                                            samples_variants[(position_ID, variant)]=[(age, VAF, error_rate, variant_depth, total_depth, (chromosome, position, ref, alt), file, cosmic_frequency, variant_type, exac_all)]
                                    else:
                                        if total_depth >= (float(mean_depth_across_panel)-(2*float(std_across_panel))):
#                                                 if total_depth >=200:
                                            if (position_ID, variant) in samples_variants.keys():
                                                samples_variants[(position_ID, variant)].append((age, VAF, error_rate, variant_depth, total_depth, (chromosome, position, ref, alt), file, cosmic_frequency, variant_type, exac_all))
                                            else:
                                                samples_variants[(position_ID, variant)]=[(age, VAF, error_rate, variant_depth, total_depth, (chromosome, position, ref, alt), file, cosmic_frequency, variant_type, exac_all)]


    #sort results for each variant by age order
    samples_variants_sorted = {}
    for k, v in samples_variants.items():
        v = sorted(v)
        samples_variants_sorted[k]=v

    ##### LOOK FOR TRAJECTORIES WITH GAPS IN THEM #######  
    samples_variants_filtered = {}
    for k, v in samples_variants_sorted.items():
        list_of_detected_variants= v
        variant = k[1]
#         print(k)
        position_information = v[0][5]
        chromosome = position_information[0]
        position = int(position_information[1])
        ref = position_information[2]
        youngest_age_variant_detected = v[0][0]
        oldest_age_variant_detected = v[-1][0]

        #--- variants detected at a single, middle timepoint ------------------------
        # No growth rate can be estimated from one detection, but under any monotonic
        # trajectory the clone must have been at least as large as the observed VAF at
        # either the preceding or the following timepoint.  Taking the larger of the two
        # resulting probabilities gives a conservative bound on the chance of having
        # missed it, with no assumption about the growth rate.  The first and last
        # timepoints are exempt: a declining clone has no earlier constraint and an
        # emerging clone has no later constraint.
        if len(v) == 1:
            ages_in_order = sorted(ages_with_samples_processed)
            position_of_detection = ages_in_order.index(v[0][0])
            if 0 < position_of_detection < (len(ages_in_order)-1):
                VAF_detected = v[0][1]
                chance_either_side = []
                try:
                    for neighbouring_age in (ages_in_order[position_of_detection-1], ages_in_order[position_of_detection+1]):
                        neighbouring_sample = retrieve_sample_name_at_age(neighbouring_age, sample_name, sample_ages)
                        neighbouring_depth = retrieve_depth_position(sample_name, neighbouring_sample, chromosome, position)
                        chance_either_side.append(np.exp(-VAF_detected*float(neighbouring_depth)))
                except (IndexError, KeyError, FileNotFoundError, TypeError, UnboundLocalError):
                    chance_either_side = [] #if the depth cannot be retrieved, keep the variant
                if len(chance_either_side)==2 and max(chance_either_side) < single_timepoint_chance_threshold:
                    # print(k[1]+' excluded: detected only at a middle timepoint, chance of missing it either side = '+str(max(chance_either_side)))
                    continue


        oldest_age_position_in_ages_list = ages_with_samples_processed.index(oldest_age_variant_detected)
        youngest_age_position_in_ages_list = ages_with_samples_processed.index(youngest_age_variant_detected) #position of youngest age detected in ages_with_samples_processed list

        expected_number_of_ages = oldest_age_position_in_ages_list - youngest_age_position_in_ages_list+1
        number_of_ages_detected = len(v)
        total_ages_missing = expected_number_of_ages - number_of_ages_detected

        proportion_of_ages_missing = total_ages_missing/expected_number_of_ages

        if total_ages_missing <0:
            samples_variants_filtered[k]=list_of_detected_variants

        else: #if there are timepoints missing when you might expect to see them...
            #look to see if should exclude the whole trajectory based on the number missing (if >1/3 missing and total chance of missing is <5%) (i.e. keep them if chance of missing is high (and they were missed))
            ages_missing = []
            ages_detected = [i[0] for i in v]
            for i in ages_with_samples_processed[youngest_age_position_in_ages_list: oldest_age_position_in_ages_list]:
                if i not in ages_detected:
                    ages_missing.append(i)

#             print('ages missing = ', ages_missing)
            
            excluded = 0
            if proportion_of_ages_missing >= 1/3:
                chance_of_missing_list = []
                for age in ages_missing:
                    sample_timepoint_name = retrieve_sample_name_at_age(age, sample_name, sample_ages) #sample_ages is a list of timepoint names and their ages
                    total_read_depth = retrieve_depth_position(sample_name, sample_timepoint_name, chromosome, position)  #read depth at that position in the missing sample

                    for i in ages_detected:
                        if i<age:
                            previous_age = i
                        if i>age:
                            next_age = i
                            break

                    previous_position_in_ages_detected_list = ages_detected.index(previous_age)
                    previous_VAF = v[previous_position_in_ages_detected_list][1]

                    next_position_in_ages_detected_list = ages_detected.index(next_age)
                    next_VAF = v[next_position_in_ages_detected_list][1]

                    growth_rate = growth_rate_calculation(previous_VAF, next_VAF, previous_age, next_age)
#                     growth_rate = ((next_VAF-previous_VAF)/previous_VAF)/(next_age-previous_age)
                    years_since_previous_timepoint = age-previous_age

                    expected_VAF, expected_reads = expected_VAF_and_reads(previous_VAF, growth_rate, years_since_previous_timepoint, total_read_depth)
                    chance_of_missing = np.exp(-expected_reads)
                    chance_of_missing_list.append(chance_of_missing)
#                     print('previous VAF = ', previous_VAF)
#                     print('next VAF = ', next_VAF)
#                     print('expected VAF = ', expected_VAF)
#                     print('chance of missing = ', chance_of_missing)

#                 print('chance of missing list', chance_of_missing_list)
                total_chance_of_missing = np.prod(chance_of_missing_list)
#                 print('total chance of missing = ', total_chance_of_missing)

                if total_chance_of_missing <0.05:
                    # print(k[1]+' trajectory completely excluded becuase >=1/3 timepoints missing and chance of this is '+str(total_chance_of_missing))
                    excluded+=1

#             print('trajectory not excluded')
#             print()
#             print('ages detected = ', ages_detected)    

            if excluded ==0: #i.e. whole trajectory hasn't been excluded, look to see if earlier timepoints should be
                list_of_detected_variants = list_of_detected_variants
                for age in sorted(ages_missing): #e.g. [70, 72]...
#                     print('age = ', age)
                    sample_timepoint_name = retrieve_sample_name_at_age(age, sample_name, sample_ages) #sample_ages is a list of timepoint names and their ages (sample_timepoint_name = e.g. C92_002_s7) = needed to retrieve depth at that position
                    total_read_depth = retrieve_depth_position(sample_name, sample_timepoint_name, chromosome, position)  #read depth at that position in the missing sample
#                     print('sample timepoint name =', sample_timepoint_name)

                    for i in ages_detected:
                        if i<age:
                            previous_age = i
                        if i>age:
                            next_age = i
                            break
#                     print('age = ', age)
#                     print('ages detected = ', ages_detected)
#                     print('previous age = ', previous_age)
#                     print('next age = ', next_age)

                    if previous_age in ages_detected: #might not be there id the samples missing are 2 consecutive timepoints and it was removed in the previous iteration
                        previous_position_in_ages_detected_list = ages_detected.index(previous_age)
                        previous_VAF = v[previous_position_in_ages_detected_list][1]

    #                     print('previous_position_in_ages_detected_list = ', previous_position_in_ages_detected_list)
    #                     print('previous_VAF = ', previous_VAF)

                        next_position_in_ages_detected_list = ages_detected.index(next_age)
                        next_VAF = v[next_position_in_ages_detected_list][1]

    #                     print('next_position_in_ages_detected_list = ', next_position_in_ages_detected_list)
    #                     print('next_VAF = ', next_VAF)

                        growth_rate = growth_rate_calculation(previous_VAF, next_VAF, previous_age, next_age)
                        years_since_previous_timepoint = age-previous_age

    #                     print('growth rate = ', growth_rate)
    #                     print('years_since_previous_timepoint = ', years_since_previous_timepoint)

                        expected_VAF, expected_reads = expected_VAF_and_reads(previous_VAF, growth_rate, years_since_previous_timepoint, total_read_depth)
                        chance_of_missing = np.exp(-expected_reads)

    #                     print('expected VAF = ', expected_VAF)
    #                     print('chance_of_missing = ', chance_of_missing)

                        if chance_of_missing<0.05:
                            list_of_detected_variants = list_of_detected_variants[next_position_in_ages_detected_list:] #i.e. start the list at the position of the next VAF
                            ages_detected = ages_detected[next_position_in_ages_detected_list:] #start the list of ages detected at the next detected sample
                            # print('samples before '+sample_timepoint_name+' for '+k[1]+' excluded due to low chance of missing '+sample_timepoint_name)
    #                         print('new ages detected = ', ages_detected)
    #                         print('new list_of_detected_variants = ', list_of_detected_variants)
    #                     print()

                if len(list_of_detected_variants)>0: #keep whatever is left of the list
                    samples_variants_filtered[k]=list_of_detected_variants
            
    #Remove trajectories if none of the variants have a VAF >4%
    samples_variants_filtered2 = {}
    for k, v in samples_variants_filtered.items():
        variant = k[1]
        gene = variant.split(' ')[0]
        AA_change = variant.split(' ')[1]
        variant_type = v[0][8]
        cosmic_frequency = v[0][7]
        timepoints_list = v
        timepoints_list = sorted(timepoints_list)
        
        timepoints_with_VAF_4 = 0
        for i in timepoints_list:
            VAF = i[1]
            if VAF >0.04:
                timepoints_with_VAF_4+=1
        if timepoints_with_VAF_4>0:
            samples_variants_filtered2[k]=timepoints_list
        
        if timepoints_with_VAF_4 == 0:
            if gene == 'NPM1':
                if variant_type.split(' ')[0]== 'frameshift':#if none of the timepoints have VAF>4%, rescue them if NPM1 288 frameshift or if variant is in COSMIC haem 1 or more times
                    amino_acid = int(AA_change.split('*')[0].split('fs')[0].split('.')[1][1:-1])
                    if amino_acid == 288:
                        samples_variants_filtered2[k]=timepoints_list
                    else:
                        if cosmic_frequency>0:
                            samples_variants_filtered2[k]=timepoints_list
                else:
                    if cosmic_frequency>0:
                        samples_variants_filtered2[k]=timepoints_list
            elif cosmic_frequency>0:
                samples_variants_filtered2[k]=timepoints_list
            # else:
            #     print(k[1]+' filtered out as no timepoints >4% VAF')

    return samples_variants_filtered2, sorted(ages_with_samples_processed)

#### mCAs

In [ ]:
def retrieve_mCAs(sample_name, sample_filenames):
        
    #retrieve the variants
    samples_variants = {}
    ages_with_samples_processed = []
    
    if len(sample_filenames[sample_name])>0:
#         print(sample_name)
        file = sample_filenames[sample_name][0]
        with open(file) as csvfile:
            readreader = csv.reader(csvfile, delimiter = '\t')
            row_count=0
            for row in readreader:
                if row_count>0:
                    sample_timepoint = row[0]
                    age = sample_ages[sample_timepoint]
                    mCA = row[2]
                    cell_fraction = row[3]
                    if row[3]not in ['not processed', 'contaminated?']:
                        ages_with_samples_processed.append(age)
                        if mCA in samples_variants.keys():
                            samples_variants[mCA].append((age, float(cell_fraction)))
                        else:
                            samples_variants[mCA]=[(age, float(cell_fraction))] 

                row_count+=1
                
    #sort results for each variant by age order
    samples_variants_sorted = {}
    for k, v in samples_variants.items():
        v = sorted(v)
        samples_variants_sorted[k]=v
                
    return samples_variants

In [ ]:
#create a list of the variants and trajectory timepoints that have already been inferred to be real (pre- common COSMIC filtering)
sample_variant_timepoints={}

for sample, variants in sample_trajectories_p_value_filtered_common_removed.items():
    sample_variant_timepoints[sample]={}
    for variant_cosmic, trajectory in variants.items():
        position_ID = variant_cosmic[0]
        variant = variant_cosmic[1] #e.g. SF3B1 p.K141K
        sample_variant_timepoints[sample][position_ID, variant]=[]
        for i in trajectory:
            sample_variant_timepoints[sample][position_ID, variant].append(i[0]) #i.e. append the age

In [ ]:
#now filter the SNV, indel and mCA calls to exclude any likely error trajectories
sample_ages_processed_SNVs = {}
sample_ages_processed_indels = {}

sample_variant_trajectories_with_germline = {}
sample_indels_trajectories_with_germline = {}
sample_mCA_trajectories = {}

for sample in cases_and_controls.keys():
    # print(sample)
    samples_variants, SNVs_ages_with_samples_processed = filter_variants_and_trajectories_with_holes(sample, max_exac, max_strand_bias, min_pos_read, sample_filenames, 
                                                                                                trajectory_p_value_threshold, sample_variant_timepoints)
    samples_indels, indels_ages_with_samples_processed = filter_indels_and_trajectories_with_holes(sample, max_exac, max_strand_bias, min_pos_read, sample_filenames, sample_variant_timepoints)
    sample_mCAs = retrieve_mCAs(sample, sample_filenames_mCAs)
    
    sample_variant_trajectories_with_germline[sample]=samples_variants
    sample_indels_trajectories_with_germline[sample]=samples_indels
    sample_mCA_trajectories[sample]=sample_mCAs
    sample_ages_processed_SNVs[sample]= SNVs_ages_with_samples_processed
    sample_ages_processed_indels[sample]= indels_ages_with_samples_processed
    # print()

## STEP 3: FILTER OUT LIKELY GERMLINE VARIANTS

 - Frequency in EXAC is < 0.001
 - RSID = '-'
 - If any timepoints have VAFs that are incompatible with germline sequencing error

In [ ]:
for sample, samples_variants in sample_variant_trajectories_with_germline.items():
    
    # Filter out variants if they have an RSID or if frequency in ExAC is >0.01
    germline_variants_filtered_out_due_to_ExAC_RSID = {}
    
    for variant, trajectory in samples_variants.items():
        germline = 0
        SNP_ID = 0
        VAFs = []
        for timepoint in trajectory:
            VAF = float(timepoint[1])
            VAFs.append(VAF)
            exac_all = float(timepoint[9])
            RSID = timepoint[10]
            if exac_all >0.001:
                germline+=1
            if RSID != '-': #i.e. if there is an RSID
                SNP_ID+=1
        if SNP_ID>0:
#             print(np.mean(VAFs))
            if np.mean(VAFs)>0.4: #(i.e. if the average VAF across all timepoints is >0.4 and there is a SNP ID) (prevents calling low VAF variants with SNP IDs as automatically germline)
                germline+=1
            # else:
            #     print()
            #     print(sample)
            #     print(variant)
            #     print(np.mean(VAFs))
            #     print()

In [ ]:
def Merge(dict1, dict2, dict3):
    return{**dict1, **dict2, **dict3}

In [ ]:
#filter the possible germline SNVs
germline_variants = {}
non_germline_variants = {}

for sample, samples_variants in sample_variant_trajectories_with_germline.items():
    # print(sample)
    
    # Filter out variants if they have an RSID or if frequency in ExAC is >0.01
    germline_variants_filtered_out_due_to_ExAC_RSID = {}
    
    for variant, trajectory in samples_variants.items():
        germline = 0
        SNP_ID = 0
        VAFs = []
        for timepoint in trajectory:
            VAF = float(timepoint[1])
            VAFs.append(VAF)
            exac_all = float(timepoint[9])
            RSID = timepoint[10]
            if exac_all >0.001:
                germline+=1
            if RSID != '-': #i.e. if there is an RSID
                SNP_ID+=1
        if SNP_ID>0:
            if np.mean(VAFs)>0.4: #(i.e. if the average VAF across all timepoints is >0.4 and there is a SNP ID) (prevents calling low VAF variants with SNP IDs as automatically germline)
                germline+=1
        if germline>0:
            germline_variants_filtered_out_due_to_ExAC_RSID[variant]=trajectory 
            
    # print('filtered out due to ExAC or RSID:')
    # print(germline_variants_filtered_out_due_to_ExAC_RSID.keys())
    
    # Filter out variants if they are observed at >40% in >5% of controls and in COSMIC <=5 times
    germline_variants_filtered_out_due_to_commonality = {}
    
    for variant, trajectory in samples_variants.items():
        if variant not in germline_variants_filtered_out_due_to_ExAC_RSID.keys():
            if variant in germline_variants_to_exclude.keys():
                germline_variants_filtered_out_due_to_commonality[variant]=trajectory 
            
    # print('filtered out due to commonality in controls at >40%:')
    # print(germline_variants_filtered_out_due_to_commonality.keys())
    
    # Filter out probable germline variant trajectories (if all timepoints have VAFs that are compatible with germline sequencing error (i.e. keep trajectory if 1 or more timepoints incompatible with germline))
    samples_variants_non_germline_binomial = {}
    germline_variants_filtered_out_due_to_binomial_p_value = {}

    for variant, trajectory in samples_variants.items():
        non_germline_timepoints = 0
        if variant not in germline_variants_filtered_out_due_to_ExAC_RSID.keys(): #i.e. already filtered out due to RSID or ExAC
            if variant not in germline_variants_filtered_out_due_to_commonality.keys():
                for timepoint in trajectory:
                    variant_depth = timepoint[3]
                    total_depth = timepoint[4]
                    VAF = timepoint[1]

                    if VAF<0.7: #for heterozygous germline_variants
                        variance_VAF = 0.5*(1/np.sqrt(total_depth))
                        z_score = (0.5-VAF)/variance_VAF
                        p_value_binomial_germline = scipy.stats.norm.sf(abs(z_score))
                        COSMIC_haem = timepoint[8]
                        if (VAF<0.5) and (p_value_binomial_germline < 0.025):  #i.e. if VAF larger/smaller than expected if it was germline+error (i.e. it is unlikely to be germline)
                            non_germline_timepoints+=1
                        if COSMIC_haem >= 5:
                            non_germline_timepoints+=1

                    else: #for homozygous germline variants
                        variance_VAF = 0.5*(1/np.sqrt(total_depth))
                        z_score = (1.0-VAF)/variance_VAF
                        p_value_binomial_germline = scipy.stats.norm.sf(abs(z_score))
                        COSMIC_haem = timepoint[8]
                        if (VAF<1.0) and (p_value_binomial_germline < 0.025):  #i.e. if VAF larger/smaller than expected if it was germline+error (i.e. it is unlikely to be germline)
                            non_germline_timepoints+=1
                        if COSMIC_haem >= 5:
                            non_germline_timepoints+=1 

                if len(trajectory)>=2:
                    if non_germline_timepoints >=2:
                        samples_variants_non_germline_binomial[variant]=trajectory
                    else:
                        germline_variants_filtered_out_due_to_binomial_p_value[variant]=trajectory
                else:
                    samples_variants_non_germline_binomial[variant]=trajectory
                
    # print('filtered out due to binomial p value:')
    # print(germline_variants_filtered_out_due_to_binomial_p_value.keys())
         
    germline_filtered_out = Merge(germline_variants_filtered_out_due_to_ExAC_RSID, germline_variants_filtered_out_due_to_commonality, germline_variants_filtered_out_due_to_binomial_p_value)
    remaining_variants = samples_variants_non_germline_binomial
    
    germline_variants[sample]=germline_filtered_out
    non_germline_variants[sample]=remaining_variants
        
    # print()

In [ ]:
#filter the possible germline indels
germline_indels = {}
non_germline_indels = {}

for sample, samples_indels in sample_indels_trajectories_with_germline.items():
    # print(sample)
    
    # Filter out variants if frequency in ExAC is >0.01
    germline_indels_filtered_out_due_to_ExAC = {}
    
    for indel, trajectory in samples_indels.items():
        germline = 0
        for timepoint in trajectory:
            exac_all = float(timepoint[9])
            if exac_all >0.001:
                germline+=1
        if germline>0:
            germline_indels_filtered_out_due_to_ExAC[indel]=trajectory 
            
    # print('filtered out due to ExAC:')
    # print(germline_indels_filtered_out_due_to_ExAC.keys())
    
    # Filter out variants if they are observed at >40% in >5% of controls and in COSMIC <=5 times
    germline_variants_filtered_out_due_to_commonality = {}
    
    for variant, trajectory in samples_variants.items():
        if variant not in germline_indels_filtered_out_due_to_ExAC.keys():
            if variant in germline_indels_to_exclude.keys():
                germline_variants_filtered_out_due_to_commonality[variant]=trajectory 
            
    # print('filtered out due to commonality in controls at >40%:')
    # print(germline_variants_filtered_out_due_to_commonality.keys())
    
    # Filter out probable germline variant trajectories (if all timepoints have VAFs that are compatible with germline sequencing error (i.e. keep trajectory if 1 or more timepoints incompatible with germline))
    samples_indels_non_germline_binomial = {}
    germline_indels_filtered_out_due_to_binomial_p_value = {}

    for indel, trajectory in samples_indels.items():
        non_germline_timepoints = 0
        if indel not in germline_indels_filtered_out_due_to_ExAC: #i.e. already filtered out due to RSID or ExAC
            if indel not in germline_variants_filtered_out_due_to_commonality.keys():
                for timepoint in trajectory:
                    variant_depth = timepoint[3]
                    total_depth = timepoint[4]
                    VAF = timepoint[1]

                    if VAF<0.7: #for heterozygous germline_variants
                        variance_VAF = 0.5*(1/np.sqrt(total_depth))
                        z_score = (0.5-VAF)/variance_VAF
                        p_value_binomial_germline = scipy.stats.norm.sf(abs(z_score))
                        COSMIC_haem = float(timepoint[7])
                        if (VAF<0.5) and (p_value_binomial_germline < 0.05):  #i.e. if VAF larger/smaller than expected if it was germline+error (i.e. it is unlikely to be germline)
                            non_germline_timepoints+=1
                        if COSMIC_haem > 5:
                            non_germline_timepoints+=1

                    else: #for homozygous germline variants
                        variance_VAF = 0.5*(1/np.sqrt(total_depth))
                        z_score = (1.0-VAF)/variance_VAF
                        p_value_binomial_germline = scipy.stats.norm.sf(abs(z_score))
                        COSMIC_haem = float(timepoint[7])
                        if (VAF<1.0) and (p_value_binomial_germline < 0.05):  #i.e. if VAF larger/smaller than expected if it was germline+error (i.e. it is unlikely to be germline)
                            non_germline_timepoints+=1
                        if COSMIC_haem > 5:
                            non_germline_timepoints+=1 

                if non_germline_timepoints > 0:
                    samples_indels_non_germline_binomial[indel]=trajectory
                else:
                    germline_indels_filtered_out_due_to_binomial_p_value[indel]=trajectory
         
    germline_filtered_out = Merge(germline_indels_filtered_out_due_to_ExAC, germline_variants_filtered_out_due_to_commonality, germline_indels_filtered_out_due_to_binomial_p_value)
    remaining_indels = samples_indels_non_germline_binomial
    
    germline_indels[sample]=germline_filtered_out
    non_germline_indels[sample]=remaining_indels
        
    # print('filtered out due to binomial p value:')
    # print(germline_indels_filtered_out_due_to_binomial_p_value.keys())
    # print()

## STEP 5: EXCLUDE LIKELY CONTAMINATED TIMEPOINTS

- samples identified by looking at the trajectories

In [ ]:
#Timepoints excluded from all analyses because the library does not appear to represent the individual it is
#labelled with.  In each case the evidence is the allele fractions at germline polymorphic sites: in an
#uncontaminated library these sit at ~0, ~0.5 or ~1, whereas a mixture of two individuals produces
#intermediate values that are all explained by a single mixture fraction.
samples_to_exclude = [
    'C92_041_s2',   #~85% of the library derives from another individual: the 7 heterozygous sites each fall to 0.06-0.10, homozygous sites shared with the other individual stay at 1.0, and the DNMT3A p.R882H clone is diluted from ~0.15 to 0.02
    'C92_044_s4',   #199 variants called at this timepoint against 18-23 at the individual's other five timepoints, i.e. a degraded or mixed library
    'C92_015_s6',   #mixture signature: 16 intermediate allele fractions against 2-4 at the other timepoints, and TET2 p.I1762V rises from ~0.50 to 0.70
    'C92_054_s3',   #mixture signature: 18 intermediate allele fractions against 3 at the other timepoints
    'CNTRL_203_s5', #mixture signature, ~60% foreign DNA: six sites fit a single mixture fraction to within 0.02, and TET2 p.L1721W appears at 0.31 where the individual is homozygous reference at every other timepoint
    ]

In [ ]:
non_germline_variants_final = {}
non_germline_indels_final = {}

for k, v in non_germline_variants.items():
    sample_variant_trajectories = {}
    for variant, trajectory in v.items():
        for timepoint in trajectory:
            file_name = timepoint[7]
            sample_name = os.path.basename(file_name).split('_')[0]+'_'+os.path.basename(file_name).split('_')[1]+'_'+os.path.basename(file_name).split('_')[2]
            if sample_name not in samples_to_exclude:
                if variant in sample_variant_trajectories.keys():
                    sample_variant_trajectories[variant].append(timepoint)
                else:
                    sample_variant_trajectories[variant]=[timepoint]
    non_germline_variants_final[k]=sample_variant_trajectories
    
for k, v in non_germline_indels.items():
    sample_indel_trajectories = {}
    for indel, trajectory in v.items():
        for timepoint in trajectory:
            file_name = timepoint[6]
            sample_name = os.path.basename(file_name).split('_')[0]+'_'+os.path.basename(file_name).split('_')[1]+'_'+os.path.basename(file_name).split('_')[2]
            if sample_name not in samples_to_exclude:
                if indel in sample_indel_trajectories.keys():
                    sample_indel_trajectories[indel].append(timepoint)
                else:
                    sample_indel_trajectories[indel]=[timepoint]
    non_germline_indels_final[k]=sample_indel_trajectories

## Variants reviewed individually

Seven variants were reviewed by hand after the automated filtering and are dealt with in the cell
below, before any output is written.

**Two were reclassified as germline.** Their somatic or germline origin was ambiguous from the
sequencing data alone, and an orthogonal method using DNA methylation (Fonseca et al., 2025)
indicated a germline origin. They are moved from the somatic table to the germline table; their VAFs
are unchanged.

**Five were excluded.** Two are the same event called twice: an insertion reported by the indel
caller and, at the same position, a substitution reported by the SNV caller. The beta-binomial error
model tests each position independently and has no knowledge of a neighbouring indel, so it cannot
recognise these as duplicates. The other three were detected at a single early timepoint and never
again. Each reached >10% VAF at one timepoint, where the error model calls a variant real without
computing a p-value; that missing p-value is read in as 0, which satisfies the p < 1e-10 rescue
condition automatically and pulls in the neighbouring sub-threshold timepoint. No threshold in the
pipeline distinguishes this from a genuine call.


In [ ]:
#--- variants reviewed individually after the automated filtering --------------------------
# See the note above.  Listed as (sample, gene, amino acid change, reason).  This is applied here,
# before anything is written, so that the per-timepoint files, the trajectory plots and the combined
# call tables all agree.

variants_reclassified_as_germline = [
    ('CNTRL_179', 'ASXL1', 'p.A709T',
     'DNA methylation data indicate a germline origin in this individual'),
    ('CNTRL_192', 'TET2', 'p.Q1542_Q1548del',
     'DNA methylation data indicate a germline origin; the sub-heterozygous VAF reflects reduced probe hybridisation efficiency across the 22bp deletion'),
]

variants_excluded_after_review = [
    ('C92_018', 'WT1', 'p.R385G',
     'same position and same variant reads as the WT1 p.R385Gfs*5 insertion: the same event called twice'),
    ('C92_023', 'ASXL1', 'p.G643R',
     'reads mismapped from the ASXL1 p.G643Rfs*15 insertion 2bp away (present at ~46% VAF): the same event called twice'),
    ('C92_018', 'ASXL1', 'p.S1422T',
     'detected at the first timepoint only, called without a p-value because VAF >10%, and absent at every later timepoint'),
    ('C92_018', 'DDX41', 'p.V408D',
     'detected at the first timepoint only, rescued by a >10% VAF call at the second timepoint, and absent thereafter'),
    ('CNTRL_162', 'RUNX1', 'p.G411R',
     'detected at the first timepoint only, rescued by a >10% VAF call at the second timepoint, and absent thereafter'),
]

def matching_variant_keys(trajectories, gene, AA_change): #trajectories are keyed (position ID, 'GENE p.XXX')
    return [key for key in trajectories.keys() if key[1]==gene+' '+AA_change]

#move the reclassified variants from the somatic trajectories into the germline trajectories
for sample_name, gene, AA_change, reason in variants_reclassified_as_germline:
    moved = 0
    for somatic, germline in ((non_germline_variants_final, germline_variants), (non_germline_indels_final, germline_indels)):
        if sample_name not in somatic:
            continue
        for key in matching_variant_keys(somatic[sample_name], gene, AA_change):
            germline.setdefault(sample_name, {})[key] = somatic[sample_name].pop(key)
            moved += 1
    print(sample_name+' '+gene+' '+AA_change+': reclassified as germline ('+str(moved)+' trajectory) - '+reason)

#remove the variants judged to be errors or duplicate calls
for sample_name, gene, AA_change, reason in variants_excluded_after_review:
    removed = 0
    for somatic in (non_germline_variants_final, non_germline_indels_final):
        if sample_name not in somatic:
            continue
        for key in matching_variant_keys(somatic[sample_name], gene, AA_change):
            del somatic[sample_name][key]
            removed += 1
    print(sample_name+' '+gene+' '+AA_change+': excluded ('+str(removed)+' trajectory) - '+reason)


## STEP 6: PLOT TRAJECTORIES

File stages:
- ..... SNVs .....
- sample_trajectories = all variants (including errors)
- sample_trajectories_p_value_filtered = variants that are called as real at 1 or more timepoint with p-value <1e-10
- sample_trajectories_p_value_filtered_common_removed = variants seen in >5% of controls removed (and not in COSMIC)
- sample_variant_trajectories_with_germline = likely error trajectories removed
- germline_variants = germline variants removed
- non_germline_variants_final = non-germline variants, with suspected contaminated samples removed
- ..... INDELS .....
- sample_indels_trajectories_with_germline = likely error trajectories removed
- germline_indels = germline indels removed
- non_germline_indels_final = non-germline indels, with suspected contaminated samples removed
- .....mCAs .....
- sample_mCA_trajectories = mCA trajectories
- ..... GERMLINE SNVS .....
- germline_variants = germline variants only
- ..... GERMLINE INDELS .....
- germline_indels = germline indels only

In [ ]:
def plot_trajectories(sample_name, samples_variants, sample_indels, sample_mCAs, case_or_control, ages_with_samples_processed, samples_to_exclude):
    #Plot data
    fig, ax1 = plt.subplots(1, 1, figsize = (16, 6))
    
    scattersize = 120
    
    diagnosis_age = sample_diagnosis_age[sample_name] 
    
    #check if any samples excluded from this sample:
    ages_to_exclude = []
    for i in samples_to_exclude:
        sample_ID = i.split('_')[0]+'_'+i.split('_')[1]
        if sample_ID == sample_name:
            excluded_age = sample_ages[i]
            ages_to_exclude.append(excluded_age)

    ages_list = []
    variant_list = []

    possible_timepoints = cases_and_controls[sample_name]
    for i in possible_timepoints:
        time = sample_ages[i]
        if time not in ages_to_exclude:
            ages_list.append(time)
        
    ages_list = sorted(ages_list)

    variant_VAFs_by_age = {}
    
    classes_plotted = {}

    for k, v in samples_variants.items():
        variant_VAFs_by_age[k[1]]={}
        v = sorted(v) #sort VAFs by age
        x = []
        y = []
        errors = []
        labels = []
        colors = []
        variant_name = k[1]
        gene = variant_name.split(' ')[0]
        mutation_class = mutation_classes[gene]

        if mutation_class in classes_plotted.keys():
            mutation_color = extra_color_classes[mutation_class][classes_plotted[mutation_class]]
            classes_plotted[mutation_class]+=1
        else:
            mutation_color = mutation_class_colors[mutation_class]
            classes_plotted[mutation_class]=0       

        for timepoint in v:
            x.append(float(timepoint[0]))#age
            y.append(float(timepoint[1])) #VAF
            labels.append(float(timepoint[2])) #p-value
            error_rate = float(timepoint[6])
            errors.append(error_rate)
            colors.append(mutation_color)
            variant_VAFs_by_age[k[1]][float(timepoint[0])]=float(timepoint[1])

        ax1.errorbar(x, y, fmt = 'o', yerr = errors, capsize = 4.5, capthick = 3, elinewidth = 3, lw = 0, ecolor = colors[0], 
                     markerfacecolor = colors[0], ms = 11, markeredgecolor = colors[0], zorder = 10, linestyle = '-')
        ax1.plot(x, y, lw = 3.5, label = k[1], color = colors[0])

        print('p-values = ', labels)
        
#         for a, b, c in zip(x, y, labels):
#             ax1.annotate("{:.2e}".format(c), xy = (a, b), xytext = (2, 2), textcoords = 'offset points', zorder = 50, fontsize = 16)
            
    n = 0
    for k, v in sample_indels.items():
        variant_VAFs_by_age[k[1]]={}
        v = sorted(v) #sort VAFs by age
        x = []
        y = []
        errors = []
        colors = []
        indel_name = k[1]
        gene = indel_name.split(' ')[0]
        mutation_class = mutation_classes[gene]

        if mutation_class in classes_plotted.keys():
            mutation_color = extra_color_classes[mutation_class][classes_plotted[mutation_class]]
            classes_plotted[mutation_class]+=1
        else:
            mutation_color = mutation_class_colors[mutation_class]
            classes_plotted[mutation_class]=0

        for timepoint in v:
            x.append(float(timepoint[0]))#age
            y.append(float(timepoint[1])) #VAF
            errors.append(float(timepoint[2]))
            colors.append(mutation_color)
            variant_VAFs_by_age[k[1]][float(timepoint[0])]=float(timepoint[1])

        try:
            ax1.errorbar(x, y, fmt = 'o', yerr = errors, capsize = 4.5, capthick = 3, elinewidth = 3, lw = 0, ecolor = colors[0], 
                         markerfacecolor = colors[0], ms = 11, markeredgecolor = colors[0], zorder = 10, linestyle = '-')
        except ValueError:
            print(errors)

        ax1.plot(x, y, lw = 3.5, label = k[1], color = colors[0])
    
    
    if len(sample_mCAs)>0:
        mCAs_plotted = 0
        n = 0
        for k, v in sample_mCAs.items():
            variant_VAFs_by_age[k]={}
            v = sorted(v) #sort VAFs by age
            x = []
            y = []
            colors = []
            labels = []
            for timepoint in v:
                if timepoint[1]!=0:
                    x.append(float(timepoint[0]))#age
                    y.append(float(timepoint[1])) #VAF
                    variant_VAFs_by_age[k][float(timepoint[0])]=float(timepoint[1])
                    if mCAs_plotted == 0:
                        colors.append(mutation_class_colors['mCA'])
                    else:
                        colors.append(extra_color_classes['mCA'][n])

            ax1.scatter(x, y, s = scattersize, color = colors)
            ax1.plot(x, y, lw = 3.5, label = k, color = colors[0])
            mCAs_plotted+=1

    all_variants_age = []
    for var in samples_variants.keys():
        variants_age = [var[1]]
        for age in ages_list:
            if age in ages_with_samples_processed:
                if age in variant_VAFs_by_age[var[1]].keys():
                    variants_age.append(str(float('%.2g' %((variant_VAFs_by_age[var[1]][age])*100)))+' %')                
                else:
                    variants_age.append(str(0)+' %')
            else:
                variants_age.append('not sequenced')
        all_variants_age.append(variants_age)
        
    for var in sample_indels.keys():
        variants_age = [var[1]]
        for age in ages_list:
            if age in ages_with_samples_processed:
                if age in variant_VAFs_by_age[var[1]].keys():
                    variants_age.append(str(float('%.2g' %((variant_VAFs_by_age[var[1]][age])*100)))+' %')                
                else:
                    variants_age.append(str(0)+' %')
            else:
                variants_age.append('not sequenced')
        all_variants_age.append(variants_age)
        
    if len(sample_mCAs)>0:
        for var in sample_mCAs.keys():
            variants_age = [var]
            for age in ages_list:
                if age in ages_with_samples_processed:
                    if age in variant_VAFs_by_age[var].keys():
                        variants_age.append(str(float('%.2g' %((variant_VAFs_by_age[var][age])*100)))+' %')                
                    else:
                        variants_age.append(str(0)+' %')
                else:
                    variants_age.append('not processed')
            all_variants_age.append(variants_age)

    fontsizing = 30
            
    if case_or_control == 'case':
        ax1.text(diagnosis_age, 1.7, 'AML \n diagnosis', ha = 'center', fontsize = fontsizing)
        ax1.plot([diagnosis_age, diagnosis_age], [0.0000005, 1.0], color = c1, lw = 3, linestyle = '--')
    else:
        ax1.text(diagnosis_age, 1.7, 'AML \n diagnosis', ha = 'center', fontsize = 1, color = 'white')
        ax1.plot([diagnosis_age, diagnosis_age], [0.0000005, 1.0], color = 'white', lw = 1, linestyle = '--', zorder = 0)

    for time in ages_with_samples_processed:
        if time not in ages_to_exclude:
            ax1.plot([time, time], [0.0000005, 1.0], color = grey2, lw = 3.5, linestyle = ':', zorder = 0)

    ax1.set_yscale('log')
    ax1.set_ylim(0.0003, 1.0)
    
    xmin, xmax = ax1.get_xlim()
    print('xmin = ', xmin)
    print('xmax = ', xmax)

    #Only show the required axis lines
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)

    for axis in ['bottom','left']:
        ax1.spines[axis].set_linewidth(2)

    for axis in ['bottom','left']:
        ax1.spines[axis].set_color(grey3)

    if len(variant_VAFs_by_age)>0:
        ax1.legend(fontsize = 24, loc='upper left', bbox_to_anchor=(1.02, 1.02))
        
    ax1.set_title(sample_name, fontsize = 15, y= 1.02)

    y_major_ticks = [0.001, 0.01, 0.1, 1.0]
    y_major_tick_labels = ["0.1%", "1%", "10%", "100%"]
    ax1.set_yticks(y_major_ticks)
    ax1.set_yticklabels(y_major_tick_labels, fontsize = fontsizing)
    ax1.yaxis.set_tick_params(width=2.5, color = grey3, length = 8, which = 'major')
    ax1.yaxis.set_tick_params(width=2, color = grey3, length = 6, which = 'minor')

    ax1.grid(axis = 'y', which = 'both', zorder = 0, linestyle = ':', lw = 1.1)

    ax1.set_xlabel('age', fontsize = fontsizing)
    ax1.set_ylabel('VAF', fontsize = fontsizing)

    plt.xticks(fontsize = fontsizing)

    # ax1.set_xlim(59.5, 67.5)

    ax1.xaxis.set_major_locator(MultipleLocator(1))
    ax1.xaxis.set_tick_params(width=2.5, color = grey3, length = 10)

#     plt.tight_layout()
#     plt.savefig(sample_name+'/'+sample_name+'_SNV_trajectories_gene_specific_colors_Oct_2023.pdf', bbox_inches='tight')
#     plt.savefig('All_trajectories/'+sample_name+'_SNV_trajectories_gene_specific_colors_Oct_2023.pdf', bbox_inches='tight')
    
    if len(variant_VAFs_by_age)>0:
        row_headers = [x.pop(0) for x in all_variants_age]

        number_variants = len(row_headers)

        table = ax1.table(cellText = all_variants_age,
                         rowLabels = row_headers,
                         colLabels = ages_list,
                         cellLoc = 'right', bbox=[0.0,-((0.1*number_variants)+0.3), 1.0, (0.1*number_variants)])               
        table.set_fontsize(14)
        table.scale(1, 2.5)

        for key, cell in table.get_celld().items():
            cell.set_linewidth(2)
            cell.set_edgecolor(grey3)
        
        
    return plt.show()

In [ ]:
def plot_trajectory(sample_name, case_or_control, SNV_trajectories, indel_trajectories, mCA_trajectories, SNV_ages_processed, indel_ages_processed, samples_to_exclude):
    all_ages_with_samples_processed = []
    for i in SNV_ages_processed[sample_name]:
        if i not in all_ages_with_samples_processed:
            all_ages_with_samples_processed.append(i)
    for i in indel_ages_processed[sample_name]:
        if i not in all_ages_with_samples_processed:
            all_ages_with_samples_processed.append(i)
    # print('ages with samples processed = ', all_ages_with_samples_processed)
    plot_trajectories(sample_name, SNV_trajectories[sample_name], indel_trajectories[sample_name], mCA_trajectories[sample_name], case_or_control, all_ages_with_samples_processed, samples_to_exclude)
    return

In [ ]:
for case in cases.keys():
    print(case)
    plot_trajectory(case, 'case', non_germline_variants_final, non_germline_indels_final, sample_mCA_trajectories, sample_ages_processed_SNVs, sample_ages_processed_indels, samples_to_exclude)

In [ ]:
for control in controls.keys():
    print(control)
    plot_trajectory(control, 'control', non_germline_variants_final, non_germline_indels_final, sample_mCA_trajectories, sample_ages_processed_SNVs, sample_ages_processed_indels, samples_to_exclude)

## STEP 7: WRITE THE VARIANT CALLS TO A FILE

## Individual sample variant call files - germline and non-germline

In [ ]:
def save_variants_file(sample_name, sample_filenames, variants_to_write, sample_ages_processed_SNVs, sample_ages, samples_to_exclude, output_suffix):
    #output_suffix selects germline or non-germline output; the two differ only in the filename written
        
    #check if any samples excluded from this sample:
    ages_to_exclude = []
    for i in samples_to_exclude:
        sample_ID = i.split('_')[0]+'_'+i.split('_')[1]
        if sample_ID == sample_name:
            excluded_age = sample_ages[i]
            ages_to_exclude.append(excluded_age)
            
    list_of_files_written = []

    ###save output file ###
    if len(variants_to_write[sample_name])>0: #i.e. if there are variants detected in at least 1 timepoint
        filtered_variants_to_write_to_file = {}
        for k, data in variants_to_write[sample_name].items():
            for v in data: #each v is a timepoint
                age = v[0]
                if age not in ages_to_exclude:
                    position_info = v[5] #e.g. ('chr20', '57484420', 'C', 'T')
                    chromosome = position_info[0]
                    position = position_info[1]
                    ref = position_info[2]
                    alt = position_info[3]
                    position_label = str(chromosome)+','+str(position)+','+str(ref)+','+str(alt)
                    file = v[7] #e.g. SNV_DIR+'C92_002_s1_SNV_watson_code_DCS_MUFs_3_beta_binomial_SNV_all_variant_calls_Oct_2023.txt'
                    if file in filtered_variants_to_write_to_file.keys():
                        filtered_variants_to_write_to_file[file].append(position_label)
                    else:
                        filtered_variants_to_write_to_file[file]=[position_label]

        for file, variants in filtered_variants_to_write_to_file.items():
            list_of_files_written.append(file)
            df = pd.read_csv(file, sep = '\t')
            df = df[df['position ID'].isin(variants)]
            new_filename = file.replace('variant_calls_Oct_2023.txt', output_suffix)
            if OUT_DIR: new_filename = OUT_DIR+os.path.basename(new_filename)
    #         print(new_filename+' written to file')
            df.to_csv(new_filename, index = False, sep = '\t')

    for file in sample_filenames[sample_name]:
        if file not in list_of_files_written: #if no variants called at any timepoint, save empty variants files
            file_split = os.path.basename(file).split('_')
            sample_timepoint_name = file_split[0]+'_'+file_split[1]+'_'+file_split[2]
            df = pd.read_csv(file, sep = '\t')
            df2 = df.iloc[0:0] #i.e. an empty dataframe with just the headings
            new_filename = file.replace('variant_calls_Oct_2023.txt', output_suffix)
            if OUT_DIR: new_filename = OUT_DIR+os.path.basename(new_filename)
            # print('empty data file produced for '+sample_timepoint_name)
            df2.to_csv(new_filename, index = False, sep = '\t')
            
    return 'variant file written for '+sample_name

In [ ]:
def save_indels_file(sample_name, sample_filenames_indels, indels_to_write, sample_ages_processed_indels, sample_ages, samples_to_exclude, output_suffix):
    #output_suffix selects germline or non-germline output; the two differ only in the filename written
        
    #check if any samples excluded from this sample:
    ages_to_exclude = []
    for i in samples_to_exclude:
        sample_ID = i.split('_')[0]+'_'+i.split('_')[1]
        if sample_ID == sample_name:
            excluded_age = sample_ages[i]
            ages_to_exclude.append(excluded_age)
            
    list_of_files_written = []

    ###save output file ###
    if len(indels_to_write[sample_name])>0: #i.e. if there are indels detected in at least 1 timepoint
        filtered_indels_to_write_to_file = {}
        for k, data in indels_to_write[sample_name].items():
            for v in data: #each v is a timepoint
                age = v[0]
                if age not in ages_to_exclude:
                    position_info = v[5] #e.g. ('chr20', '57484420', 'C', 'T')
                    chromosome = position_info[0]
                    position = position_info[1]
                    ref = position_info[2]
                    alt = position_info[3]
                    position_label = str(chromosome)+','+str(position)+','+str(ref)+','+str(alt)
                    file = v[6] #e.g. INDEL_DIR+'C92_002_s1_SNV_SNV_watson_code_DCS_VarDictJava_indel_variant_calls.txt'
                    if file in filtered_indels_to_write_to_file.keys():
                        filtered_indels_to_write_to_file[file].append(position_label)
                    else:
                        filtered_indels_to_write_to_file[file]=[position_label]

        for file, indels in filtered_indels_to_write_to_file.items():
            list_of_files_written.append(file)
            df = pd.read_csv(file, sep = '\t')
            df['start']=df['start'].astype(str)
            df['position ID']=df['chromosome']+','+df['start']+','+df['REF']+','+df['ALT']
            df = df[df['position ID'].isin(indels)]
            df['start']=df['start'].astype(int)
            new_filename = file.replace('indel_variant_calls.txt', output_suffix)
            if OUT_DIR: new_filename = OUT_DIR+os.path.basename(new_filename)
    #         print(new_filename+' written to file')
            df.to_csv(new_filename, index = False, sep = '\t')

    for file in sample_filenames_indels[sample_name]:
        if file not in list_of_files_written: #if no indels called at any timepoint, save empty indels files
            file_split = os.path.basename(file).split('_')
            sample_timepoint_name = file_split[0]+'_'+file_split[1]+'_'+file_split[2]
            df = pd.read_csv(file, sep = '\t')
            df['start']=df['start'].astype(str)
            df['position ID']=df['chromosome']+','+df['start']+','+df['REF']+','+df['ALT']
            df['start']=df['start'].astype(int)
            df2 = df.iloc[0:0] #i.e. an empty dataframe with just the headings
            new_filename = file.replace('indel_variant_calls.txt', output_suffix)
            if OUT_DIR: new_filename = OUT_DIR+os.path.basename(new_filename)
            # print('empty data file produced for '+sample_timepoint_name)
            df2.to_csv(new_filename, index = False, sep = '\t')
            
    return 'indel variant file written for '+sample_name

### Germline variants (individual files per sample)

In [ ]:
#SNVs
for sample_name in cases_and_controls.keys():
    save_variants_file(sample_name, sample_filenames, germline_variants, sample_ages_processed_SNVs, sample_ages, samples_to_exclude, 'germline_variant_calls_2026_post_processed.txt')

#Indels
for sample_name in cases_and_controls.keys():
    save_indels_file(sample_name, sample_filenames_indels, germline_indels, sample_ages_processed_SNVs, sample_ages, samples_to_exclude, 'germline_indel_variant_calls_2026_post_processed.txt')

### Non-germline variants (individual files per sample)

In [ ]:
for sample_name in cases_and_controls.keys():
    save_variants_file(sample_name, sample_filenames, non_germline_variants_final, sample_ages_processed_SNVs, sample_ages, samples_to_exclude, 'non-germline_variant_calls_2026_post_processed.txt')

for sample_name in cases_and_controls.keys():
    save_indels_file(sample_name, sample_filenames_indels, non_germline_indels_final, sample_ages_processed_indels, sample_ages, samples_to_exclude, 'non-germline_indel_variant_calls_2026_post_processed.txt')

## Overall variant call files

In [ ]:
#Make dictionaries
sample_file = 'Data_files/UKCTOCS_samples_processed_information.csv'

sample_timepoint_IDs = {}
sample_timepoint_IDs_shortname = {}
sample_details = {}

with open(sample_file) as csvfile:
    read_reader = csv.reader(csvfile)
    row_count =0
    
    for row in read_reader:
        if row_count>0:
            sample_ID = row[1]
            sample_name = sample_ID.split('_')[0]+'_'+sample_ID.split('_')[1]
            timepoint = 's'+sample_ID.split('_')[2][1:].zfill(2)
            age_sample = row[8]
            age_diagnosis = row[7]
            time_to_diagnosis = row[5]
            grouping = row[6]
            matched_sample = row[9]

            if sample_name not in sample_timepoint_IDs:
                sample_timepoint_IDs[sample_name]=[sample_ID]
            else:
                sample_timepoint_IDs[sample_name].append(sample_ID)

            sample_timepoint_IDs_shortname[sample_ID]=timepoint

            sample_details[sample_ID] = {'age_at_sample_taken': age_sample,
                                        'age_at_diagnosis': age_diagnosis,
                                         'time_to_diagnosis': time_to_diagnosis,
                                         'grouping': grouping,
                                         'matched_sample': matched_sample}
        row_count+=1

In [ ]:
def get_matched_sample_name(sample_ID):
    return sample_details[sample_ID]['matched_sample']

def get_age_at_diagnosis(sample_ID):
    return sample_details[sample_ID]['age_at_diagnosis']

def get_age_sample_taken(sample_ID):
    return sample_details[sample_ID]['age_at_sample_taken']

def get_time_to_diagnosis(sample_ID):
    return sample_details[sample_ID]['time_to_diagnosis']

def get_timepoint_shortname(sample_ID):
    return sample_timepoint_IDs_shortname[sample_ID]

In [ ]:
# Create combined SNV dataframe (for germline or non-germline)
def create_dataframe(sample_name, sample_timepoint, samples_to_exclude, call_type): #call_type is 'non-germline' or 'germline'
    #SNVs...
    if sample_timepoint not in samples_to_exclude:
        df = pd.read_csv(SNV_DIR+sample_timepoint+'_'+'SNV_watson_code_DCS_MUFs_3_beta_binomial_SNV_all_'+call_type+'_variant_calls_2026_post_processed.txt', sep='\t')
        df['sample name']=sample_timepoint
        df['matched_sample']=df['sample name'].apply(get_matched_sample_name)
        df['age_sample_taken']=df['sample name'].apply(get_age_sample_taken)
        df['months_to_diagnosis']=df['sample name'].apply(get_time_to_diagnosis)
        df['age_at_AML_diagnosis']=df['sample name'].apply(get_age_at_diagnosis)
        return df

def create_sample_dataframe_dictionary(sample_name, samples_to_exclude, call_type):
    sample_df = {}
    for timepoint_sample in sample_timepoint_IDs[sample_name]:
        df = create_dataframe(sample_name, timepoint_sample, samples_to_exclude, call_type)
        sample_df[timepoint_sample]=df

    return pd.concat(sample_df.values(), ignore_index=True)

In [ ]:
# Create combined indel dataframe (for germline or non-germline)
def create_indels_dataframe(sample_name, sample_timepoint, samples_to_exclude, call_type): #call_type is 'non-germline' or 'germline'
    #indels...
    if sample_timepoint not in samples_to_exclude:
        if sample_timepoint not in ['C92_009_s5', 'CNTRL_004_s4', 'CNTRL_004_s6', 'CNTRL_168_s3', 'CNTRL_195_s1', 'CNTRL_196_s4']: #don't have indels anyway
            try:
                df = pd.read_csv(INDEL_DIR+sample_timepoint+'_'+'SNV_SNV_watson_code_DCS_VarDictJava_'+call_type+'_indel_variant_calls_2026_post_processed.txt', sep='\t')
                df['sample name']=sample_timepoint
                df['matched_sample']=df['sample name'].apply(get_matched_sample_name)
                df['age_sample_taken']=df['sample name'].apply(get_age_sample_taken)
                df['months_to_diagnosis']=df['sample name'].apply(get_time_to_diagnosis)
                df['age_at_AML_diagnosis']=df['sample name'].apply(get_age_at_diagnosis)
            except FileNotFoundError:
                df = pd.read_csv(INDEL_DIR+sample_timepoint+'_'+'SNV_watson_code_DCS_VarDictJava_'+call_type+'_indel_variant_calls_2026_post_processed.txt', sep='\t')
                df['sample name']=sample_timepoint
                df['matched_sample']=df['sample name'].apply(get_matched_sample_name)
                df['age_sample_taken']=df['sample name'].apply(get_age_sample_taken)
                df['months_to_diagnosis']=df['sample name'].apply(get_time_to_diagnosis)
                df['age_at_AML_diagnosis']=df['sample name'].apply(get_age_at_diagnosis)
            return df

def create_indels_sample_dataframe_dictionary(sample_name, samples_to_exclude, call_type):
    sample_df = {}
    for timepoint_sample in sample_timepoint_IDs[sample_name]:
        df = create_indels_dataframe(sample_name, timepoint_sample, samples_to_exclude, call_type)
        sample_df[timepoint_sample]=df

    return pd.concat(sample_df.values(), ignore_index=True)

### Non-germline variants (all samples together)

Creates:
- `UKCTOCS_non-germline_variants_calls_SNVs_indels_mCAs.csv` (unrounded ages)
- `Somatic_SNV_indel_FLT3_calls.csv` (rounded ages)
- `UKCTOCS_non-germline_variants_calls_SNVs_indels_mCAs_rounded_ages.csv` (rounded ages)
- `UKCTOCS_germline_variants_calls_SNV_indel_panel.csv`

In [ ]:
def chromosome_without_arm(chromosome):
    return re.sub(r'[pq]$', '', str(chromosome))

def variant_type(mCA):
    return mCA.split(' ')[1]

def chromosome_label(mCA):
    return 'chr'+mCA.split(' ')[0]

def get_sample_name(sample_ID):
    return sample_ID.split('_')[0]+'_'+sample_ID.split('_')[1]

In [ ]:
all_SNVs_dataframes = {}
all_indels_dataframes = {}
mCAs_df = {}

#SNVs
for sample_name in sample_timepoint_IDs.keys():
    concat_df = create_sample_dataframe_dictionary(sample_name, samples_to_exclude, 'non-germline')
    all_SNVs_dataframes[sample_name]=concat_df

all_SNVs_data = pd.concat(all_SNVs_dataframes.values(), ignore_index=True)
all_SNVs_data = all_SNVs_data.rename(columns={"call": "MLE call"})

all_SNVs_data = all_SNVs_data.rename(columns={"total depth": "total_depth", "variant depth": "variant_depth", "VAF": "VAF (cell fraction for mCAs)"})

all_SNVs_data_trimmed = all_SNVs_data[['sample name', 'age_sample_taken', 'months_to_diagnosis', 'age_at_AML_diagnosis', 'matched_sample',
                                       'chromosome', 'start', 'end', 'REF', 'ALT', 'total_depth', 'variant_depth', 'VAF (cell fraction for mCAs)',
                                       'intronic_exonic', 'variant_type', 'gene',
                                       'transcript', 'exon', 'cdNA', 'AA_change', 'exonic_function', 'fitting method', 'iteration called at',
                                       'p-value', 'MLE call', 'position final error rate', 'position final delta', 'total variants called at position',
                                       'cosmic_ID', 'cosmic_total', 'cosmic_haem_lymphoid',
                                       'cosmic_sites', 'exac_all', 'gnomad_all', 'clinvar_allele_id', 'clinvar_dn',
                                       'clinvar_disdb', 'clinvar_rev', 'clin_sig']].copy()

#Indels
for sample_name in sample_timepoint_IDs.keys():
    concat_indels_df = create_indels_sample_dataframe_dictionary(sample_name, samples_to_exclude, 'non-germline')
    all_indels_dataframes[sample_name]=concat_indels_df

all_indels_data = pd.concat(all_indels_dataframes.values(), ignore_index=True)
all_indels_data = all_indels_data.rename(columns={"VAF": "VAF (cell fraction for mCAs)"})

for i in all_SNVs_data_trimmed.columns: #add the columns in the SNV dataframe to the indel dataframe
    if i not in all_indels_data.columns:
        all_indels_data[i]=''

all_indels_data_trimmed = all_indels_data[['sample name', 'age_sample_taken', 'months_to_diagnosis', 'age_at_AML_diagnosis', 'matched_sample',
                                       'chromosome', 'start', 'end', 'REF', 'ALT', 'total_depth', 'variant_depth', 'VAF (cell fraction for mCAs)',
                                       'intronic_exonic', 'variant_type', 'gene',
                                       'transcript', 'exon', 'cdNA', 'AA_change', 'exonic_function', 'fitting method', 'iteration called at',
                                       'p-value', 'MLE call', 'position final error rate', 'position final delta', 'total variants called at position',
                                       'cosmic_ID', 'cosmic_total', 'cosmic_haem_lymphoid',
                                       'cosmic_sites', 'exac_all', 'gnomad_all', 'clinvar_allele_id', 'clinvar_dn',
                                       'clinvar_disdb', 'clinvar_rev', 'clin_sig']].copy()

#mCAs
for k, v in sample_filenames_mCAs.items():
    if len(v)>0:
        sample = k
        filename = v[0]
        df = pd.read_csv(filename, sep = '\t')
        df = df.rename(columns={"Timepoint": "sample name"})
        df['matched_sample']=df['sample name'].apply(get_matched_sample_name)
        df['age_sample_taken']=df['sample name'].apply(get_age_sample_taken)
        df['months_to_diagnosis']=df['sample name'].apply(get_time_to_diagnosis)
        df['age_at_AML_diagnosis']=df['sample name'].apply(get_age_at_diagnosis)
        mCAs_df[sample]=df
        
all_mCAs_df = pd.concat(mCAs_df.values(), ignore_index=True)

mCA_caller_calls = pd.read_csv('Data_files/Somatic_mCA_calls.csv')
mCA_caller_calls['event type'] = mCA_caller_calls['mCA type'].str.replace('CN-LOH', 'CNLOH', regex=False)
mCA_caller_calls['individual'] = mCA_caller_calls['Sample name'].str.rsplit('_s', n=1).str[0]
mCA_caller_calls['chromosome (no arm)'] = mCA_caller_calls['Chromosome'].str.replace(r'[pq]$', '', regex=True)

all_mCAs_df['variant_type'] = all_mCAs_df['mCA'].apply(variant_type)
all_mCAs_df['chromosome'] = all_mCAs_df['mCA'].apply(chromosome_label)
all_mCAs_df = all_mCAs_df.drop(columns=['mCA'])
all_mCAs_df = all_mCAs_df.rename(columns={"cell fraction": "VAF (cell fraction for mCAs)"})

#this runs after 'mCA' has been split into variant_type/chromosome and 'cell fraction' renamed,
#so it uses those columns rather than the original ones
for index, row in all_mCAs_df.iterrows():
    event_type = row['variant_type']
    individual = row['sample name'].rsplit('_s', 1)[0]
    event = mCA_caller_calls[(mCA_caller_calls['individual']==individual) &
                            (mCA_caller_calls['chromosome (no arm)']==chromosome_without_arm(row['chromosome'])) &
                            (mCA_caller_calls['event type']==event_type)]
    if len(event)>0:
        all_mCAs_df.loc[index, 'start'] = int(event.iloc[0]['Start'])
        all_mCAs_df.loc[index, 'end'] = int(event.iloc[0]['End'])
        this_timepoint = event[event['Sample name']==row['sample name']]
        if len(this_timepoint)>0:
            all_mCAs_df.loc[index, 'VAF (cell fraction for mCAs)'] = this_timepoint.iloc[0]['Cell fraction']

for i in all_SNVs_data_trimmed.columns: #add the columns in the SNV dataframe to the mCA dataframe
    if i not in all_mCAs_df.columns:
        all_mCAs_df[i]=''

all_mCAs_data_trimmed = all_mCAs_df[['sample name', 'age_sample_taken', 'months_to_diagnosis', 'age_at_AML_diagnosis', 'matched_sample',
                                       'chromosome', 'start', 'end', 'REF', 'ALT', 'total_depth', 'variant_depth', 'VAF (cell fraction for mCAs)',
                                       'intronic_exonic', 'variant_type', 'gene',
                                       'transcript', 'exon', 'cdNA', 'AA_change', 'exonic_function', 'fitting method', 'iteration called at',
                                       'p-value', 'MLE call', 'position final error rate', 'position final delta', 'total variants called at position',
                                       'cosmic_ID', 'cosmic_total', 'cosmic_haem_lymphoid',
                                       'cosmic_sites', 'exac_all', 'gnomad_all', 'clinvar_allele_id', 'clinvar_dn',
                                       'clinvar_disdb', 'clinvar_rev', 'clin_sig']].copy()


In [ ]:
SNVs_and_indels_and_mCAs_df = pd.concat([all_SNVs_data_trimmed, all_indels_data_trimmed, all_mCAs_data_trimmed])
SNVs_and_indels_and_mCAs_df['timepoint_name'] = SNVs_and_indels_and_mCAs_df['sample name'].apply(get_timepoint_shortname)
SNVs_and_indels_and_mCAs_df['sample full name'] = SNVs_and_indels_and_mCAs_df['sample name'].apply(get_sample_name)
SNVs_and_indels_and_mCAs_df = SNVs_and_indels_and_mCAs_df.sort_values(['sample full name', 'timepoint_name', 'VAF (cell fraction for mCAs)'], ascending = [True, True, False])

SNVs_and_indels_and_mCAs_df = SNVs_and_indels_and_mCAs_df.drop(columns=['sample full name', 'timepoint_name'])

#combined somatic call table (SNVs, indels and mCAs) - this is what the figure notebooks and the optimiser read
combined_somatic_calls = SNVs_and_indels_and_mCAs_df.drop(columns=error_model_columns)

#genomic coordinates are whole numbers: concatenating frames with differing dtypes promotes them to float
for position_column in ['start', 'end']:
    combined_somatic_calls[position_column] = combined_somatic_calls[position_column].astype('int64')
combined_somatic_calls.to_csv((OUT_DIR or 'Data_files/')+'UKCTOCS_non-germline_variants_calls_SNVs_indels_mCAs.csv', index = False)

#ages and event timings are reduced to whole numbers in the released tables, to protect participant confidentiality
age_columns = ['age_sample_taken', 'months_to_diagnosis', 'age_at_AML_diagnosis']

def completed_whole_units(value): #a person aged 73.73 years has completed 73 years, so these are rounded down rather than to the nearest whole number
    if pd.isna(value) or str(value).strip()=='':
        return value
    return int(math.floor(float(value)))

combined_somatic_calls_rounded_ages = combined_somatic_calls.copy()
for age_column in age_columns:
    combined_somatic_calls_rounded_ages[age_column] = combined_somatic_calls_rounded_ages[age_column].apply(completed_whole_units)
combined_somatic_calls_rounded_ages.to_csv((OUT_DIR or 'Data_files/')+'UKCTOCS_non-germline_variants_calls_SNVs_indels_mCAs_rounded_ages.csv', index = False)

#Supplementary Table 6: somatic SNV and indel calls only, with rounded ages (mCAs are in Supplementary Table 7, produced by the mCA caller)
supplementary_table_6 = combined_somatic_calls_rounded_ages[~combined_somatic_calls_rounded_ages['variant_type'].isin(mCA_variant_types)].copy()
supplementary_table_6 = supplementary_table_6.rename(columns=supplementary_table_column_names)
supplementary_table_6.to_csv((OUT_DIR or 'Data_files/')+'Somatic_SNV_indel_FLT3_calls.csv', index = False)
supplementary_table_6

In [ ]:
combined_somatic_calls

### Germline variants (all samples together)

Creates `UKCTOCS_germline_variants_calls_SNV_indel_panel.csv`

In [ ]:
#SNVs
all_germline_SNVs_dataframes = {}

for sample_name in sample_timepoint_IDs.keys():
    concat_df = create_sample_dataframe_dictionary(sample_name, samples_to_exclude, 'germline')
    all_germline_SNVs_dataframes[sample_name]=concat_df

all_germline_SNVs_data = pd.concat(all_germline_SNVs_dataframes.values(), ignore_index=True)
all_germline_SNVs_data = all_germline_SNVs_data.rename(columns={"call": "MLE call"})

all_SNPs_data = all_germline_SNVs_data.rename(columns={"total depth": "total_depth", "variant depth": "variant_depth", "VAF": "VAF (cell fraction for mCAs)"})

all_SNPs_data_trimmed = all_SNPs_data[['sample name', 'age_sample_taken', 'months_to_diagnosis', 'age_at_AML_diagnosis', 'matched_sample',
                                       'chromosome', 'start', 'end', 'REF', 'ALT', 'total_depth', 'variant_depth', 'VAF (cell fraction for mCAs)',
                                       'intronic_exonic', 'variant_type', 'gene',
                                       'transcript', 'exon', 'cdNA', 'AA_change', 'exonic_function', 'fitting method', 'iteration called at',
                                       'p-value', 'MLE call', 'position final error rate', 'position final delta', 'total variants called at position',
                                       'cosmic_ID', 'cosmic_total', 'cosmic_haem_lymphoid',
                                       'cosmic_sites', 'exac_all', 'gnomad_all', 'clinvar_allele_id', 'clinvar_dn',
                                       'clinvar_disdb', 'clinvar_rev', 'clin_sig']].copy()

#indels
all_germline_indels_dataframes = {}

for sample_name in sample_timepoint_IDs.keys():
    concat_germline_indels_df = create_indels_sample_dataframe_dictionary(sample_name, samples_to_exclude, 'germline')
    all_germline_indels_dataframes[sample_name]=concat_germline_indels_df

all_germline_indels_data = pd.concat(all_germline_indels_dataframes.values(), ignore_index=True)
all_germline_indels_data = all_germline_indels_data.rename(columns={"VAF": "VAF (cell fraction for mCAs)"}) #add the columns in the SNV dataframe to the indel datafram

for i in all_SNPs_data_trimmed.columns:
    if i not in all_germline_indels_data.columns:
        all_germline_indels_data[i]=''

all_germline_indels_data_trimmed = all_germline_indels_data[['sample name', 'age_sample_taken', 'months_to_diagnosis', 'age_at_AML_diagnosis', 'matched_sample',
                                       'chromosome', 'start', 'end', 'REF', 'ALT', 'total_depth', 'variant_depth', 'VAF (cell fraction for mCAs)',
                                       'intronic_exonic', 'variant_type', 'gene',
                                       'transcript', 'exon', 'cdNA', 'AA_change', 'exonic_function', 'fitting method', 'iteration called at',
                                       'p-value', 'MLE call', 'position final error rate', 'position final delta', 'total variants called at position',
                                       'cosmic_ID', 'cosmic_total', 'cosmic_haem_lymphoid',
                                       'cosmic_sites', 'exac_all', 'gnomad_all', 'clinvar_allele_id', 'clinvar_dn',
                                       'clinvar_disdb', 'clinvar_rev', 'clin_sig']].copy()


In [ ]:
all_germline_df = pd.concat([all_SNPs_data_trimmed, all_germline_indels_data_trimmed])
all_germline_df['timepoint_name'] = all_germline_df['sample name'].apply(get_timepoint_shortname)
all_germline_df['sample full name'] = all_germline_df['sample name'].apply(get_sample_name)
all_germline_df = all_germline_df.sort_values(['sample full name', 'timepoint_name', 'VAF (cell fraction for mCAs)'], ascending = [True, True, False])

all_germline_df = all_germline_df.drop(columns=['sample full name', 'timepoint_name'])

#germline variant calls from the SNV/ indel panel - deposited with the EGA submission
germline_calls = all_germline_df.drop(columns=error_model_columns)

#genomic coordinates are whole numbers: concatenating frames with differing dtypes promotes them to float
for position_column in ['start', 'end']:
    germline_calls[position_column] = germline_calls[position_column].astype('int64')
germline_calls.to_csv((OUT_DIR or 'Data_files/')+'UKCTOCS_germline_variants_calls_SNV_indel_panel.csv', index = False)
all_germline_df